In [ ]:
import sys
print(f"Python version: {sys.version}")
print("Basic import successful!")

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, chi2, RFE, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight

try:
    import xgboost as xgb
    print("✓ XGBoost imported successfully")
except Exception as e:
    print(f"⚠ XGBoost not available. Run: brew install libomp")
    xgb = None

try:
    import lightgbm as lgb
    print("✓ LightGBM imported successfully")
except Exception as e:
    print(f"⚠ LightGBM not available: {str(e)[:50]}...")
    lgb = None

try:
    from catboost import CatBoostClassifier
    print("✓ CatBoost imported successfully")
except Exception as e:
    print(f"⚠ CatBoost not available: {str(e)[:50]}...")
    CatBoostClassifier = None

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, classification_report, matthews_corrcoef
)

try:
    import shap
    print("✓ SHAP imported successfully")
except Exception as e:
    print(f"⚠ SHAP not available: {str(e)[:50]}...")
    shap = None

try:
    from lime import lime_tabular
    print("✓ LIME imported successfully")
except Exception as e:
    print(f"⚠ LIME not available: {str(e)[:50]}...")
    lime_tabular = None

import joblib
import pickle

# Statistical analysis
from scipy import stats
from scipy.stats import zscore
from datetime import datetime


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')


print("✓ Core libraries imported successfully!")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")



## 2. Define File Paths and Configuration

In [ ]:
BASE_DIR = Path('/Users/roya/Desktop/Thesis')
DATA_DIR = BASE_DIR / 'Data set'
OUTPUT_DIR = BASE_DIR / 'Thesis_Project' / 'ML'
PART_B_FILE = DATA_DIR / 'Part B' / '2023' / 'MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv'
PART_D_FILE = DATA_DIR / 'Medicare Part D Prescribers - by Provider and Drug' / '2023' / 'MUP_DPR_RY25_P04_V10_DY23_NPIBN.csv'
OIG_FILE = DATA_DIR / 'OIG.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(exist_ok=True)

print("✓ File paths configured")
print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

## 3. Data Loading & Initial Exploration

### 3.1 Load Medicare Part B Data

In [ ]:
print("Loading Medicare Part B 2023 data...")
part_b_df = pd.read_csv(PART_B_FILE, low_memory=False)

print(f"Part B Data Shape: {part_b_df.shape}")
print(f"\nMemory Usage: {part_b_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
part_b_df.head()

### 3.2

In [ ]:
us_states = ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 
             'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 
             'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 
             'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 
             'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 
             'DC', 'PR', 'VI', 'GU', 'AS', 'MP']

before_filter_b = len(part_b_df)
part_b_df = part_b_df[part_b_df['Rndrng_Prvdr_State_Abrvtn'].isin(us_states)]
after_filter_b = len(part_b_df)
removed_b = before_filter_b - after_filter_b

print(f"Part B: Removed {removed_b:,} non-U.S. records ({removed_b/before_filter_b*100:.2f}%)")
print(f"Part B: Remaining records: {after_filter_b:,}")
print(f"Part B Data Shape after filtering: {part_b_df.shape}")

### 3.3

In [ ]:
print("PROVIDER LOCATION CONSISTENCY CHECK - PART B")

provider_location_check_b = part_b_df.groupby('Rndrng_NPI').agg({
    'Rndrng_Prvdr_State_Abrvtn': 'nunique',
    'Rndrng_Prvdr_Zip5': 'nunique',
    'Rndrng_Prvdr_RUCA': 'nunique'
}).rename(columns={
    'Rndrng_Prvdr_State_Abrvtn': 'Unique_States',
    'Rndrng_Prvdr_Zip5': 'Unique_Zipcodes',
    'Rndrng_Prvdr_RUCA': 'Unique_RUCA'
})

total_providers = len(provider_location_check_b)
single_state = (provider_location_check_b['Unique_States'] == 1).sum()
multiple_states = (provider_location_check_b['Unique_States'] > 1).sum()

single_zip = (provider_location_check_b['Unique_Zipcodes'] == 1).sum()
multiple_zip = (provider_location_check_b['Unique_Zipcodes'] > 1).sum()

single_ruca = (provider_location_check_b['Unique_RUCA'] == 1).sum()
multiple_ruca = (provider_location_check_b['Unique_RUCA'] > 1).sum()

print(f"\nTotal unique providers: {total_providers:,}")
print(f"\n{'Field':<20} {'Single Value':<20} {'Multiple Values':<20}")
print(f"{'State':<20} {single_state:,} ({single_state/total_providers*100:.2f}%)    {multiple_states:,} ({multiple_states/total_providers*100:.2f}%)")
print(f"{'Zipcode':<20} {single_zip:,} ({single_zip/total_providers*100:.2f}%)    {multiple_zip:,} ({multiple_zip/total_providers*100:.2f}%)")
print(f"{'RUCA Code':<20} {single_ruca:,} ({single_ruca/total_providers*100:.2f}%)    {multiple_ruca:,} ({multiple_ruca/total_providers*100:.2f}%)")
print("MIN AND MAX STATISTICS FOR UNIQUE VALUES")
print(f"\n{'Field':<20} {'Min':<15} {'Max':<15} {'Mean':<15}")
print(f"{'Unique_States':<20} {provider_location_check_b['Unique_States'].min():<15} {provider_location_check_b['Unique_States'].max():<15} {provider_location_check_b['Unique_States'].mean():<15.2f}")
print(f"{'Unique_Zipcodes':<20} {provider_location_check_b['Unique_Zipcodes'].min():<15} {provider_location_check_b['Unique_Zipcodes'].max():<15} {provider_location_check_b['Unique_Zipcodes'].mean():<15.2f}")
print(f"{'Unique_RUCA':<20} {provider_location_check_b['Unique_RUCA'].min():<15} {provider_location_check_b['Unique_RUCA'].max():<15} {provider_location_check_b['Unique_RUCA'].mean():<15.2f}")
print("DETAILED PROVIDER LOCATION TABLE")

provider_location_check_b['Has_Multiple_Locations'] = (
    (provider_location_check_b['Unique_States'] > 1) |
    (provider_location_check_b['Unique_Zipcodes'] > 1) |
    (provider_location_check_b['Unique_RUCA'] > 1)
)


print(f"\nShowing all {len(provider_location_check_b):,} providers:")
display(provider_location_check_b)


if multiple_states > 0 or multiple_zip > 0 or multiple_ruca > 0:
   
    print("PROVIDERS WITH INCONSISTENT LOCATIONS")

    multi_location_providers = provider_location_check_b[
        provider_location_check_b['Has_Multiple_Locations']
    ]
    print(f"\nTotal providers with multiple locations: {len(multi_location_providers):,}")
    display(multi_location_providers)



In [ ]:
print(part_b_df.columns)

remove = ['Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_MI', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_RUCA_Desc', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'HCPCS_Desc', ]

keep = [ 'Rndrng_NPI', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_Zip5', Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Ent_Cd', '', 'Rndrng_Prvdr_Type', 'HCPCS_Cd', 'HCPCS_Drug_Ind', 'Place_Of_Srvc', 'Tot_Benes',	'Tot_Srvcs', 'Tot_Bene_Day_Srvcs', 'Avg_Sbmtd_Chrg',	'Avg_Mdcr_Alowd_Amt', 	'Avg_Mdcr_Pymt_Amt', 	'Avg_Mdcr_Stdzd_Amt'   ]

In [ ]:
part_b_df['Rndrng_Prvdr_Ent_Cd'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Cntry'].unique()

In [ ]:
part_b_df['HCPCS_Drug_Ind'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Crdntls'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Mdcr_Prtcptg_Ind'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_RUCA'].unique()

### 3.4

In [ ]:
provider_summary = part_b_df.groupby('Rndrng_NPI').agg({
    'Rndrng_Prvdr_Type': lambda x: list(x.unique()),
    'HCPCS_Cd': lambda x: list(x.unique()),
    'HCPCS_Drug_Ind': lambda x: list(x.unique())
}).reset_index()

provider_summary['Num_Unique_Provider_Types'] = provider_summary['Rndrng_Prvdr_Type'].apply(len)
provider_summary['Num_Unique_HCPCS_Codes'] = provider_summary['HCPCS_Cd'].apply(len)
provider_summary['Num_Unique_HCPCS_Drug_Ind'] = provider_summary['HCPCS_Drug_Ind'].apply(len)

provider_summary['Provider_Types'] = provider_summary['Rndrng_Prvdr_Type'].apply(lambda x: ', '.join(map(str, x)))
provider_summary['HCPCS_Codes'] = provider_summary['HCPCS_Cd'].apply(lambda x: ', '.join(map(str, x)))
provider_summary['HCPCS_Drug_Ind_List'] = provider_summary['HCPCS_Drug_Ind'].apply(lambda x: ', '.join(map(str, x)))

provider_summary_display = provider_summary[[
    'Rndrng_NPI',
    'Num_Unique_Provider_Types',
    'Num_Unique_HCPCS_Codes',
    'Num_Unique_HCPCS_Drug_Ind',
    'Provider_Types',
    'HCPCS_Codes',
    'HCPCS_Drug_Ind_List'
]]

print(f"\nTotal number of unique providers: {len(provider_summary_display)}")
print(f"\nFirst 20 providers:")
print(provider_summary_display.head(20).to_string(index=False))


print("SUMMARY STATISTICS")
print(f"Average provider types per provider: {provider_summary_display['Num_Unique_Provider_Types'].mean():.2f}")
print(f"Average HCPCS codes per provider: {provider_summary_display['Num_Unique_HCPCS_Codes'].mean():.2f}")
print(f"Average HCPCS Drug Ind per provider: {provider_summary_display['Num_Unique_HCPCS_Drug_Ind'].mean():.2f}")
print(f"Max provider types for a single provider: {provider_summary_display['Num_Unique_Provider_Types'].max()}")
print(f"Max HCPCS codes for a single provider: {provider_summary_display['Num_Unique_HCPCS_Codes'].max()}")
print(f"Max HCPCS Drug Ind for a single provider: {provider_summary_display['Num_Unique_HCPCS_Drug_Ind'].max()}")


provider_summary_display

In [ ]:
part_b_df['Place_Of_Srvc'].unique()

In [ ]:
part_b_df['HCPCS_Drug_Ind'].unique()

In [ ]:
print("Part B Data Info:")
part_b_df.info()

In [ ]:
part_b_df.describe()

### 3.2 Load Medicare Part D Data

### 3.2.1

In [ ]:
print("Loading Medicare Part D 2023 data...")
part_d_df = pd.read_csv(PART_D_FILE, low_memory=False)

print(f"Part D Data Shape: {part_d_df.shape}")
print(f"\nMemory Usage: {part_d_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

part_d_df.head()

### 3.2.2

In [ ]:
before_filter_d = len(part_d_df)
part_d_df = part_d_df[part_d_df['Prscrbr_State_Abrvtn'].isin(us_states)]
after_filter_d = len(part_d_df)
removed_d = before_filter_d - after_filter_d

print(f"Part D: Removed {removed_d:,} non-U.S. records ({removed_d/before_filter_d*100:.2f}%)")
print(f"Part D: Remaining records: {after_filter_d:,}")
print(f"Part D Data Shape after filtering: {part_d_df.shape}")

### 3.2.3

In [ ]:
print("PROVIDER LOCATION CONSISTENCY CHECK - PART D")

provider_location_check_d = part_d_df.groupby('Prscrbr_NPI').agg({
    'Prscrbr_State_Abrvtn': 'nunique',
    'Prscrbr_State_FIPS': 'nunique'
}).rename(columns={
    'Prscrbr_State_Abrvtn': 'Unique_States',
    'Prscrbr_State_FIPS': 'Unique_State_FIPS'
})

total_providers_d = len(provider_location_check_d)
single_state_d = (provider_location_check_d['Unique_States'] == 1).sum()
multiple_states_d = (provider_location_check_d['Unique_States'] > 1).sum()

single_fips_d = (provider_location_check_d['Unique_State_FIPS'] == 1).sum()
multiple_fips_d = (provider_location_check_d['Unique_State_FIPS'] > 1).sum()

print(f"\nTotal unique prescribers: {total_providers_d:,}")
print(f"\n{'Field':<20} {'Single Value':<20} {'Multiple Values':<20}")

print(f"{'State':<20} {single_state_d:,} ({single_state_d/total_providers_d*100:.2f}%)    {multiple_states_d:,} ({multiple_states_d/total_providers_d*100:.2f}%)")
print(f"{'State FIPS':<20} {single_fips_d:,} ({single_fips_d/total_providers_d*100:.2f}%)    {multiple_fips_d:,} ({multiple_fips_d/total_providers_d*100:.2f}%)")

print("MIN AND MAX STATISTICS FOR UNIQUE VALUES")

print(f"\n{'Field':<20} {'Min':<15} {'Max':<15} {'Mean':<15}")

print(f"{'Unique_States':<20} {provider_location_check_d['Unique_States'].min():<15} {provider_location_check_d['Unique_States'].max():<15} {provider_location_check_d['Unique_States'].mean():<15.2f}")
print(f"{'Unique_State_FIPS':<20} {provider_location_check_d['Unique_State_FIPS'].min():<15} {provider_location_check_d['Unique_State_FIPS'].max():<15} {provider_location_check_d['Unique_State_FIPS'].mean():<15.2f}")
print("DETAILED PROVIDER LOCATION TABLE")

provider_location_check_d['Has_Multiple_Locations'] = (
    (provider_location_check_d['Unique_States'] > 1) |
    (provider_location_check_d['Unique_State_FIPS'] > 1)
)

print(f"\nShowing all {len(provider_location_check_d):,} prescribers:")
display(provider_location_check_d)


if multiple_states_d > 0 or multiple_fips_d > 0:
   
    print("PRESCRIBERS WITH INCONSISTENT LOCATIONS")
 
    multi_location_prescribers = provider_location_check_d[
        provider_location_check_d['Has_Multiple_Locations']
    ]
    print(f"\nTotal prescribers with multiple locations: {len(multi_location_prescribers):,}")
    display(multi_location_prescribers)



In [ ]:
print(part_d_df.columns)

remove = ['Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_FIPS', 'Prscrbr_Type_Src',  'GE65_Sprsn_Flag', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Tot_Benes']

keep = ['Prscrbr_NPI', , 'Prscrbr_State_Abrvtn', 'Prscrbr_Type', 'Brnd_Name', 'Gnrc_Name','Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst','Tot_Benes']

Tot_30day_Fills': 'sum',  # How many monthly prescriptions the provider wrote

'Tot_Day_Suply': 'sum',  # Total number of days of medication given to patients

'Tot_Benes': 'sum',  # Number of different Medicare patients who received this drug from this provider(note: 0 means suppressed <11)
    
 how much money
'Tot_Drug_Cst': ['sum'],  # total money spent at pharmacies because of this provider’s prescriptions

### 3.2.5

In [ ]:
print("PROVIDER SUMMARY TABLE - PART D")

provider_summary_d = part_d_df.groupby('Prscrbr_NPI').agg({
    'Prscrbr_Type': lambda x: list(x.unique()), 
    'Brnd_Name': lambda x: list(x.unique()),
    'Gnrc_Name': lambda x: list(x.unique())
}).reset_index() 

provider_summary_d['Num_Unique_Provider_Types'] = provider_summary_d['Prscrbr_Type'].apply(len)
provider_summary_d['Num_Unique_Brnd_Names'] = provider_summary_d['Brnd_Name'].apply(len)
provider_summary_d['Num_Unique_Gnrc_Names'] = provider_summary_d['Gnrc_Name'].apply(len)


provider_summary_d['Provider_Types'] = provider_summary_d['Prscrbr_Type'].apply(lambda x: ', '.join(map(str, x)))
provider_summary_d['Brand_Names'] = provider_summary_d['Brnd_Name'].apply(lambda x: ', '.join(map(str, x)))
provider_summary_d['Generic_Names'] = provider_summary_d['Gnrc_Name'].apply(lambda x: ', '.join(map(str, x)))

provider_summary_d_display = provider_summary_d[[
    'Prscrbr_NPI',
    'Num_Unique_Provider_Types',
    'Num_Unique_Brnd_Names',
    'Num_Unique_Gnrc_Names',
    'Provider_Types',
    'Brand_Names',
    'Generic_Names'
]]

print(f"\nTotal number of unique providers: {len(provider_summary_d_display)}")
print(f"\nFirst 20 providers:")
print(provider_summary_d_display.head(20).to_string(index=False))

print("SUMMARY STATISTICS - PART D")
print(f"Average provider types per provider: {provider_summary_d_display['Num_Unique_Provider_Types'].mean():.2f}")
print(f"Average brand names per provider: {provider_summary_d_display['Num_Unique_Brnd_Names'].mean():.2f}")
print(f"Average generic names per provider: {provider_summary_d_display['Num_Unique_Gnrc_Names'].mean():.2f}")
print(f"Max provider types for a single provider: {provider_summary_d_display['Num_Unique_Provider_Types'].max()}")
print(f"Max brand names for a single provider: {provider_summary_d_display['Num_Unique_Brnd_Names'].max()}")
print(f"Max generic names for a single provider: {provider_summary_d_display['Num_Unique_Gnrc_Names'].max()}")

provider_summary_d_display

In [ ]:
part_d_df['Prscrbr_State_Abrvtn'].unique()

In [ ]:
part_d_df['Prscrbr_Type_Src'].unique()

In [ ]:
print("Part D Data Info:")
part_d_df.info()

### 3.3 Load OIG Exclusion List

In [ ]:
print("Loading OIG Exclusion List...")
oig_df = pd.read_csv(OIG_FILE, low_memory=False)

print(f"OIG Data Shape: {oig_df.shape}")
print(f"\nMemory Usage: {oig_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

oig_df.head()

In [ ]:
print(oig_df.columns)

In [ ]:
print("OIG Data Info:")
oig_df.info()

print("Unique values in key columns:")
print(f"Unique Last Names: {oig_df['LASTNAME'].nunique() if 'LASTNAME' in oig_df.columns else 'N/A'}")
print(f"Unique States: {oig_df['STATE'].nunique() if 'STATE' in oig_df.columns else 'N/A'}")

## 4. Data Quality Assessment

### 4.1 Missing Values Analysis

In [ ]:
def analyze_missing_values(df, df_name):
    print(f"Missing Values Analysis: {df_name}")

    missing_values = ['', ' ', '  ', 'nan', 'NaN', 'NA', 'N/A', 'null', 'NULL', 'None', 'none', '<NA>']
    df_temp = df.replace(missing_values, pd.NA)
    
    missing_count = df_temp.isnull().sum()
    missing_percent = (missing_count / len(df_temp)) * 100
    
    missing_df = pd.DataFrame({
        'Column': df_temp.columns,
        'Missing_Count': missing_count.values,
        'Missing_Percent': missing_percent.values
    })
    
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(
        'Missing_Percent', ascending=False
    ).reset_index(drop=True)
    
    if len(missing_df) == 0:
        print("✓ No missing values found!")
    else:
        print(f"\nColumns with missing values: {len(missing_df)}")
        print(missing_df)
        
        if len(missing_df) <= 20:
            fig = px.bar(
                missing_df,
                x='Column',
                y='Missing_Percent',
                title=f'Missing Values in {df_name}',
                labels={'Missing_Percent': 'Missing %'},
                text='Missing_Percent'
            )
            fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
            fig.update_layout(xaxis_tickangle=-45, height=500)
            fig.show()
    
    return missing_df

missing_part_b = analyze_missing_values(part_b_df, 'Medicare Part B 2023')
missing_part_d = analyze_missing_values(part_d_df, 'Medicare Part D 2023')
missing_oig = analyze_missing_values(oig_df, 'OIG Exclusion List')

### 4.2 Duplicate Records Check

In [ ]:
def check_duplicates(df, df_name):
    print(f"\nDuplicate Analysis: {df_name}")

    full_duplicates = df.duplicated().sum()
    print(f"Full duplicate rows: {full_duplicates} ({full_duplicates/len(df)*100:.2f}%)")
    
    return full_duplicates

dup_part_b = check_duplicates(part_b_df, 'Part B')
dup_part_d = check_duplicates(part_d_df, 'Part D')
dup_oig = check_duplicates(oig_df, 'OIG')

## 5. Data Preprocessing & Cleaning


In [ ]:
def basic_data_cleaning(df, df_name):
    
    df_clean = df.copy()
    initial_rows = len(df_clean)
    
    before_dedup = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    after_dedup = len(df_clean)
    removed_dups = before_dedup - after_dedup
    print(f"1. Removed duplicates: {removed_dups:,} rows")

    npi_cols = [col for col in df_clean.columns if 'NPI' in col.upper()]
    for npi_col in npi_cols:
        if df_clean[npi_col].dtype == 'object':
            df_clean[npi_col] = df_clean[npi_col].astype(str).str.strip()
            valid_npi = df_clean[npi_col].str.isnumeric() | df_clean[npi_col].isna()
            invalid_count = (~valid_npi).sum()
            if invalid_count > 0:
                df_clean = df_clean[valid_npi]
                print(f"2. Removed invalid NPIs in {npi_col}: {invalid_count:,} rows")

    all_nan_rows = df_clean.isnull().all(axis=1).sum()
    if all_nan_rows > 0:
        df_clean = df_clean.dropna(how='all')
        print(f"3. Removed completely empty rows: {all_nan_rows:,}")
    
    final_rows = len(df_clean)
    total_removed = initial_rows - final_rows
    missing_count = df_clean.isnull().sum().sum()
    
    print(f"\nCLEANING SUMMARY:")
    print(f"  Initial rows:     {initial_rows:,}")
    print(f"  Final rows:       {final_rows:,}")
    print(f"  Rows removed:     {total_removed:,} ({total_removed/initial_rows*100:.2f}%)")
    print(f"  Missing values:   {missing_count:,} (PRESERVED for post-aggregation imputation)")
    print()
    
    return df_clean


In [ ]:
print("Applying basic data cleaning (no imputation yet)...")
print()

part_b_clean = basic_data_cleaning(part_b_df, "Part B - Provider Services")
part_d_clean = basic_data_cleaning(part_d_df, "Part D - Prescription Drugs")
oig_clean = basic_data_cleaning(oig_df, "OIG - Exclusion List")


print("✓ Basic cleaning complete! Missing values preserved for post-aggregation imputation.")


## 6. Provider-Level Aggregation


### 6.1 Create Provider-Level Aggregations - Part B


In [ ]:
print("Creating provider-level aggregations from Part B data...")

part_b_clean['Weighted_Payment'] = part_b_clean['Tot_Srvcs'] * part_b_clean['Avg_Mdcr_Pymt_Amt']
part_b_agg = part_b_clean.groupby('Rndrng_NPI').agg({
    'Rndrng_Prvdr_Last_Org_Name': 'first',
    'Rndrng_Prvdr_First_Name': 'first',
    'Rndrng_Prvdr_State_Abrvtn': 'first',
    'Rndrng_Prvdr_Zip5': 'first',
    'Rndrng_Prvdr_Type': 'first',  
    'Rndrng_Prvdr_Ent_Cd': 'first',  
    'Rndrng_Prvdr_RUCA': 'first', 

    'Tot_Benes': 'mean',  
    'Tot_Srvcs': 'sum',
    'Tot_Bene_Day_Srvcs': 'sum',

    
    'Avg_Sbmtd_Chrg': ['mean', 'max'],      
    'Avg_Mdcr_Alowd_Amt': ['mean', 'max'], 
    'Avg_Mdcr_Pymt_Amt': 'mean',            
    'Avg_Mdcr_Stdzd_Amt': 'mean',          
    'Weighted_Payment': 'sum',              

   
    'HCPCS_Cd': 'nunique',          
    'HCPCS_Drug_Ind': 'nunique',    
    'Place_Of_Srvc': 'nunique',     
}).reset_index()

part_b_agg.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in part_b_agg.columns.values]

column_mapping = {
    'Rndrng_NPI': 'NPI',
    'Rndrng_Prvdr_Last_Org_Name_first': 'Provider_Name',
    'Rndrng_Prvdr_First_Name_first': 'Provider_FirstName',
    'Rndrng_Prvdr_State_Abrvtn_first': 'State',
    'Rndrng_Prvdr_Zip5_first': 'Zipcode',
    'Rndrng_Prvdr_Type_first': 'Provider_Type',
    'Rndrng_Prvdr_Ent_Cd_first': 'Entity_Code',
    'Rndrng_Prvdr_RUCA_first': 'RUCA_Code',

    'Tot_Benes_mean': 'Total_Beneficiaries_B',
    'Tot_Srvcs_sum': 'Total_Services_B',
    'Tot_Bene_Day_Srvcs_sum': 'Total_BeneDay_Services_B',

    'Avg_Sbmtd_Chrg_mean': 'Avg_Submitted_Charge_B',
    'Avg_Sbmtd_Chrg_max': 'Max_Submitted_Charge_B',
    'Avg_Mdcr_Alowd_Amt_mean': 'Avg_Medicare_Allowed_B',
    'Avg_Mdcr_Alowd_Amt_max': 'Max_Medicare_Allowed_B',
    'Avg_Mdcr_Pymt_Amt_mean': 'Avg_Medicare_Payment_B',
    'Avg_Mdcr_Stdzd_Amt_mean': 'Avg_Medicare_Standardized_B',
    'Weighted_Payment_sum': 'Weighted_Avg_Payment_B', 

    'HCPCS_Cd_nunique': 'Unique_HCPCS_B',
    'HCPCS_Drug_Ind_nunique': 'Service_Type_Diversity_B',  
    'Place_Of_Srvc_nunique': 'Unique_Service_Places_B',
}
part_b_agg.rename(columns=column_mapping, inplace=True)


part_b_agg['Weighted_Avg_Payment_B'] = (
    part_b_agg['Weighted_Avg_Payment_B'] / part_b_agg['Total_Services_B']
)

print(f"✓ Part B aggregated to {len(part_b_agg):,} unique providers")
print(f"  Columns: {part_b_agg.shape[1]}")
print(f"  Features: {part_b_agg.shape[1] - 7} numeric + 7 descriptive")
part_b_agg.head()


In [ ]:
min_val = part_b_agg['Unique_HCPCS_B'].min()
max_val = part_b_agg['Unique_HCPCS_B'].max()

print(f"Unique_HCPCS_B (unique HCPCS codes per provider):")
print(f"  Min : {min_val:,}")
print(f"  Max : {max_val:,}")
print(f"\nProvider(s) with Min unique HCPCS codes ({min_val}):")
print(part_b_agg.loc[part_b_agg['Unique_HCPCS_B'] == min_val,
                     ['NPI', 'Provider_Name', 'Provider_Type', 'Unique_HCPCS_B']].head())
print(f"\nProvider(s) with Max unique HCPCS codes ({max_val}):")
print(part_b_agg.loc[part_b_agg['Unique_HCPCS_B'] == max_val,
                     ['NPI', 'Provider_Name', 'Provider_Type', 'Unique_HCPCS_B']].head())


### 6.2 Create Provider-Level Aggregations - Part D

In [ ]:
print("\nCreating provider-level aggregations from Part D data...")

part_d_agg = part_d_clean.groupby('Prscrbr_NPI').agg({
    'Prscrbr_Last_Org_Name': 'first',
    'Prscrbr_First_Name': 'first',
    'Prscrbr_State_Abrvtn': 'first',
    'Prscrbr_Type': 'first',


    'Tot_Clms': 'sum',     
    'Tot_30day_Fills': 'sum',   
    'Tot_Day_Suply': 'sum',      


    'Tot_Drug_Cst': ['sum', 'mean'],  

    'Brnd_Name': 'nunique',     
    'Gnrc_Name': 'nunique',    
}).reset_index()

part_d_agg.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in part_d_agg.columns.values]

column_mapping = {
    'Prscrbr_NPI': 'NPI',
    'Prscrbr_Last_Org_Name_first': 'Prescriber_Name_D',
    'Prscrbr_First_Name_first': 'Prescriber_FirstName_D',
    'Prscrbr_State_Abrvtn_first': 'Prescriber_State_D',
    'Prscrbr_Type_first': 'Prescriber_Type_D',
    'Tot_Clms_sum': 'Total_Claims_D',
    'Tot_30day_Fills_sum': 'Total_30Day_Fills_D',
    'Tot_Day_Suply_sum': 'Total_Day_Supply_D',
    'Tot_Drug_Cst_sum': 'Total_Drug_Cost_D',
    'Tot_Drug_Cst_mean': 'Avg_Drug_Cost_Per_Claim_D',
    'Brnd_Name_nunique': 'Unique_Brand_Drugs_D',
    'Gnrc_Name_nunique': 'Unique_Generic_Drugs_D',
}
part_d_agg.rename(columns=column_mapping, inplace=True)

print(f"✓ Part D aggregated to {len(part_d_agg):,} unique prescribers")
part_d_agg.head()


In [ ]:
npi_b = set(part_b_agg['NPI'])
npi_d = set(part_d_agg['NPI'])
npi_both = npi_b & npi_d

pct_b_in_d = len(npi_both) / len(npi_b) * 100  
pct_d_in_b = len(npi_both) / len(npi_d) * 100  

print("NPI Overlap between Part B and Part D:")
print(f"  Unique NPIs in Part B           : {len(npi_b):>10,}")
print(f"  Unique NPIs in Part D           : {len(npi_d):>10,}")
print(f"  NPIs present in both            : {len(npi_both):>10,}")
print(f"  NPIs only in Part B             : {len(npi_b - npi_d):>10,}")
print(f"  NPIs only in Part D             : {len(npi_d - npi_b):>10,}")
print(f"\n  % of Part B providers in Part D : {pct_b_in_d:.2f}%")
print(f"  % of Part D providers in Part B : {pct_d_in_b:.2f}%")


### 6.3 Create Fraud Labels from OIG Data

In [ ]:
print("\nCreating fraud labels from OIG exclusion list...")

oig_npi_list = oig_clean['NPI'].dropna().astype(str).str.strip()
oig_npi_list = oig_npi_list[oig_npi_list.str.isnumeric()]
oig_npi_set = set(oig_npi_list.astype(int))
print(f"Total excluded providers in OIG: {len(oig_npi_set):,}")


print("\nMerging Part B and Part D data...")
providers_df = part_b_agg.merge(part_d_agg, on='NPI', how='outer', indicator=True)
print(f"Combined dataset: {len(providers_df):,} providers")
print(f"  - Only in Part B: {(providers_df['_merge'] == 'left_only').sum():,}")
print(f"  - Only in Part D: {(providers_df['_merge'] == 'right_only').sum():,}")
print(f"  - In both datasets: {(providers_df['_merge'] == 'both').sum():,}")


providers_df['Data_Source'] = providers_df['_merge'].map({
    'left_only': 'Part_B_Only',
    'right_only': 'Part_D_Only',
    'both': 'Both'
})
providers_df.drop('_merge', axis=1, inplace=True)

print("\nConsolidating provider information...")
providers_df['Provider_Name'] = providers_df['Provider_Name'].fillna(providers_df['Prescriber_Name_D'])
providers_df['State'] = providers_df['State'].fillna(providers_df['Prescriber_State_D'])
providers_df['Provider_Type'] = providers_df['Provider_Type'].fillna(providers_df['Prescriber_Type_D'])

print("\nCreating fraud labels...")
providers_df['Is_Fraudulent'] = providers_df['NPI'].isin(oig_npi_set).astype(int)

print(f"\nFraud Label Distribution:")
fraud_counts = providers_df['Is_Fraudulent'].value_counts()
print(f"  Non-Fraudulent (0): {fraud_counts.get(0, 0):,}")
print(f"  Fraudulent (1):     {fraud_counts.get(1, 0):,}")
print(f"  Fraud rate:         {providers_df['Is_Fraudulent'].mean()*100:.4f}%")

print(f"\nFraud distribution by data source:")
fraud_by_source = pd.crosstab(providers_df['Data_Source'], providers_df['Is_Fraudulent'], margins=True)
print(fraud_by_source)

print(f"✓ Final merged dataset: {providers_df.shape[0]:,} providers × {providers_df.shape[1]} features")
providers_df.head()

In [ ]:
print("MISSING VALUE ANALYSIS - MERGED DATASET")

missing_data = pd.DataFrame({
    'Column': providers_df.columns,
    'Missing_Count': providers_df.isnull().sum().values,
    'Missing_Percent': (providers_df.isnull().sum().values / len(providers_df) * 100)
})
missing_data = missing_data.sort_values('Missing_Percent', ascending=False)

missing_data_filtered = missing_data[missing_data['Missing_Count'] > 0]

print(f"\nTotal columns: {len(providers_df.columns)}")
print(f"Columns with missing values: {len(missing_data_filtered)}")
print(f"Columns without missing values: {len(providers_df.columns) - len(missing_data_filtered)}")

print(f"\n{'Column':<40} {'Missing Count':<15} {'Missing %':<15}")
for _, row in missing_data_filtered.iterrows():
    print(f"{row['Column']:<40} {int(row['Missing_Count']):<15,} {row['Missing_Percent']:<15.2f}")

import plotly.graph_objects as go

plot_data = missing_data_filtered.head(20) 

fig = go.Figure()

fig.add_trace(go.Bar(
    x=plot_data['Missing_Percent'],
    y=plot_data['Column'],
    orientation='h',
    marker=dict(
        color=plot_data['Missing_Percent'],
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title="Missing %")
    ),
    text=[f"{val:.2f}%" for val in plot_data['Missing_Percent']],
    textposition='auto',
))

fig.update_layout(
    title="Missing Value Percentage by Column (Merged Dataset)",
    xaxis_title="Missing Percentage (%)",
    yaxis_title="Column Name",
    height=max(400, len(plot_data) * 25),
    showlegend=False,
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()



### 6.3.1 Missing Value Analysis

### 6.4 Post-Merge Data Quality & Missing Value Handling


   - **Entity_Code**: Inferred from Provider_FirstName (397,548 values)
     - If FirstName exists → 'I' (Individual)
     - If FirstName empty/null → 'O' (Organization)
   - **Provider_FirstName**: Empty string for organizations (397,548 values)
   - **Zipcode**: State-based mode imputation (397,548 values)
   - **Part B metrics**: 0 for Part D-only providers (397,548 × 15 features)
     - Zero means "no Part B activity", not actual zero services
   - **Part D metrics**: 0 for Part B-only providers (468,667 × 13 features)
     - Zero means "no Part D activity", not actual zero prescriptions
   - **Prescriber info include Prescriber_Name_D, Prescriber_State_D, Prescriber_Type_D**: Copied from Provider info for Part B-only providers (468,667 × 3 fields)



### 6.4.1

In [ ]:
print("VERIFYING POST-MERGE IMPUTATION")

print(f"\nBefore imputation:")
print(f"  Missing values: {providers_df.isnull().sum().sum():,}")

print(f"\nAfter imputation:")
print(f"  Missing values: {providers_df.isnull().sum().sum():,}")

print(f"\nKey columns after imputation:")
print(f"  Entity_Code missing: {providers_df['Entity_Code'].isnull().sum()}")
print(f"  Provider_FirstName missing: {providers_df['Provider_FirstName'].isnull().sum()}")
print(f"  Zipcode missing: {providers_df['Zipcode'].isnull().sum()}")
print(f"  Prescriber_Name_D missing: {providers_df['Prescriber_Name_D'].isnull().sum()}")
print(f"  Prescriber_State_D missing: {providers_df['Prescriber_State_D'].isnull().sum()}")
print(f"  Prescriber_Type_D missing: {providers_df['Prescriber_Type_D'].isnull().sum()}")

print("SECTION 6.4 COMPLETE - FINAL DATASET SUMMARY")

### 6.4.2

In [ ]:
print("SEMANTIC MISSING VALUE IMPUTATION")

def impute_post_merge_missing(df):

    df_imputed = df.copy()
    imputation_log = []
    part_d_only = df_imputed.get('Data_Source', pd.Series('', index=df_imputed.index)) == 'Part_D_Only'
    part_b_only = df_imputed.get('Data_Source', pd.Series('', index=df_imputed.index)) == 'Part_B_Only'

    if 'Entity_Code' in df_imputed.columns and 'Provider_FirstName' in df_imputed.columns:
        missing_entity = df_imputed['Entity_Code'].isnull()
        if missing_entity.sum() > 0:
        
            df_imputed.loc[missing_entity & part_d_only, 'Entity_Code'] = 'Unknown'

            has_firstname = df_imputed['Provider_FirstName'].notna() & (df_imputed['Provider_FirstName'] != '')
            df_imputed.loc[missing_entity & ~part_d_only & has_firstname, 'Entity_Code'] = 'I'
            df_imputed.loc[missing_entity & ~part_d_only & ~has_firstname, 'Entity_Code'] = 'O'
            imputation_log.append(f"Entity_Code: {missing_entity.sum()} values imputed "
                                  f"('Unknown' for Part D-only; I/O heuristic for Part B providers)")

    if 'Provider_FirstName' in df_imputed.columns:
        missing_fn = df_imputed['Provider_FirstName'].isnull()
        if missing_fn.sum() > 0:
            df_imputed.loc[missing_fn & part_d_only, 'Provider_FirstName'] = 'Unknown'
            df_imputed.loc[missing_fn & ~part_d_only, 'Provider_FirstName'] = ''
            imputation_log.append(f"Provider_FirstName: {missing_fn.sum()} values imputed "
                                  f"('Unknown' for Part D-only; '' for Part B organizations)")

    for state_col in ['State', 'State_B', 'State_D']:
        if state_col in df_imputed.columns:
            missing_state = df_imputed[state_col].isnull()
            if missing_state.sum() > 0:
                df_imputed.loc[missing_state, state_col] = 'Unknown'
                imputation_log.append(f"{state_col}: {missing_state.sum()} values imputed to 'Unknown' "
                                      f"(no state data in either dataset)")

    for type_col in ['Provider_Type', 'Provider_Type_B', 'Provider_Type_D']:
        if type_col in df_imputed.columns:
            missing_type = df_imputed[type_col].isnull()
            if missing_type.sum() > 0:
                df_imputed.loc[missing_type, type_col] = 'Unknown'
                imputation_log.append(f"{type_col}: {missing_type.sum()} values imputed to 'Unknown' "
                                      f"(no specialty data in either dataset)")

    if 'Zipcode' in df_imputed.columns:
        missing_zip = df_imputed['Zipcode'].isnull()
        if missing_zip.sum() > 0:
            df_imputed.loc[missing_zip & part_d_only, 'Zipcode'] = 'Unknown'
            missing_zip_b = df_imputed['Zipcode'].isnull()  
            if missing_zip_b.sum() > 0:
                if 'State' in df_imputed.columns:
                    for state in df_imputed.loc[missing_zip_b, 'State'].unique():
                        state_mask = missing_zip_b & (df_imputed['State'] == state)
                        mode_zip = df_imputed[df_imputed['State'] == state]['Zipcode'].mode()
                        fill_val = mode_zip[0] if len(mode_zip) > 0 else '00000'
                        df_imputed.loc[state_mask, 'Zipcode'] = fill_val
                else:
                    df_imputed.loc[missing_zip_b, 'Zipcode'] = '00000'
            imputation_log.append(f"Zipcode: {missing_zip.sum()} values imputed "
                                  f"('Unknown' for Part D-only; state-based mode for Part B providers)")

    numeric_cols = df_imputed.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col not in ['NPI', 'Fraud_Flag']:
            missing_num = df_imputed[col].isnull()
            if missing_num.sum() > 0:
                df_imputed.loc[missing_num, col] = 0
                imputation_log.append(f"{col}: {missing_num.sum()} values imputed to 0 "
                                      f"(no activity / not in source dataset)")


    prescriber_crossfill = {
        'Prescriber_Name_D':      'Provider_Name',
        'Prescriber_FirstName_D': 'Provider_FirstName',
        'Prescriber_State_D':     'State',
        'Prescriber_Type_D':      'Provider_Type',
    }
    for d_col, b_col in prescriber_crossfill.items():
        if d_col in df_imputed.columns:
            missing_val = df_imputed[d_col].isnull()
            if missing_val.sum() > 0:
                if b_col in df_imputed.columns:
                    df_imputed.loc[missing_val, d_col] = df_imputed.loc[missing_val, b_col]
                still_missing = df_imputed[d_col].isnull()
                if still_missing.sum() > 0:
                    df_imputed.loc[still_missing, d_col] = 'Unknown'
                imputation_log.append(f"{d_col}: {missing_val.sum()} values imputed "
                                      f"(cross-filled from {b_col}; 'Unknown' for any remainder)")

    if 'Data_Source' in df_imputed.columns:
        missing_val = df_imputed['Data_Source'].isnull()
        if missing_val.sum() > 0:
            df_imputed.loc[missing_val, 'Data_Source'] = 'Unknown'
            imputation_log.append(f"Data_Source: {missing_val.sum()} values imputed to 'Unknown'")

    print("\nImputation Summary:")
    for log_entry in imputation_log:
        print(f"  • {log_entry}")
    if not imputation_log:
        print("  ✓ No missing values required imputation")

    return df_imputed


providers_df_clean = impute_post_merge_missing(providers_df)

final_missing = providers_df_clean.isnull().sum().sum()

print(f"FINAL VERIFICATION: {final_missing} missing values remaining")

if final_missing == 0:
    print("✓ All missing values successfully imputed!")
else:
    print(f"⚠️  Still have {final_missing} missing values")
    remaining_missing = providers_df_clean.isnull().sum()
    print("\nRemaining missing values:")
    print(remaining_missing[remaining_missing > 0])


### 6.4.3

In [ ]:
providers_df_clean.head(10)

### 6.4.4

In [ ]:
print(f"\nBefore removal: {providers_df_clean.shape}")
print(f"Columns: {len(providers_df_clean.columns)}")

redundant_cols = ['Prescriber_Name_D', 'Prescriber_FirstName_D', 'Prescriber_State_D', 'Prescriber_Type_D']


print(f"\nRemoving Part D prescriber identity columns (redundant after consolidation in Section 6.3):")
for col in redundant_cols:
    if col in providers_df_clean.columns:
        print(f"  ✗ {col}")

providers_df_clean = providers_df_clean.drop(columns=redundant_cols, errors='ignore')


providers_df = providers_df_clean.copy()

print(f"\nAfter removal: {providers_df.shape}")
print(f"Columns: {len(providers_df.columns)}")
print(f"\nRemaining columns:")
print(providers_df.columns.tolist())

print("\n✓ Redundant columns removed successfully!")

In [ ]:
providers_df.head(10)

## 7. Exploratory Data Analysis (EDA)

### 7.1 Feature Engineering

In [ ]:
_n_cols_before = providers_df.shape[1]


print("Section 7.1  Advanced Feature Engineering")
print(f"\nStarting shape  : {providers_df.shape[0]:,} rows × {providers_df.shape[1]:,} columns")
print(f"Missing values  : {providers_df.isnull().sum().sum():,}")

print("\n[1] Payment & charge ratio features ...")

providers_df['Charge_to_Allowed_Ratio'] = np.where(
    providers_df['Avg_Medicare_Allowed_B'] > 0,
    providers_df['Avg_Submitted_Charge_B'] / providers_df['Avg_Medicare_Allowed_B'],
    0
)

providers_df['Payment_to_Allowed_Ratio'] = np.where(
    providers_df['Avg_Medicare_Allowed_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Medicare_Allowed_B'],
    0
)

providers_df['MaxCharge_Ratio'] = np.where(
    providers_df['Avg_Submitted_Charge_B'] > 0,
    providers_df['Max_Submitted_Charge_B'] / providers_df['Avg_Submitted_Charge_B'],
    0
)

providers_df['Allowed_Charge_Gap'] = (
    providers_df['Avg_Submitted_Charge_B'] - providers_df['Avg_Medicare_Allowed_B']
)

providers_df['Payment_Efficiency'] = np.where(
    providers_df['Avg_Submitted_Charge_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Submitted_Charge_B'],
    0
)

providers_df['Standardized_Payment_Ratio'] = np.where(
    providers_df['Avg_Medicare_Standardized_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Medicare_Standardized_B'],
    0
)

print("[2] Service intensity & volume features ...")

providers_df['Services_per_Beneficiary_B'] = np.where(
    providers_df['Total_Beneficiaries_B'] > 0,
    providers_df['Total_Services_B'] / providers_df['Total_Beneficiaries_B'],
    0
)

providers_df['BeneDay_Per_Beneficiary_B'] = np.where(
    providers_df['Total_Beneficiaries_B'] > 0,
    providers_df['Total_BeneDay_Services_B'] / providers_df['Total_Beneficiaries_B'],
    0
)

providers_df['Services_Per_HCPCS'] = np.where(
    providers_df['Unique_HCPCS_B'] > 0,
    providers_df['Total_Services_B'] / providers_df['Unique_HCPCS_B'],
    0
)


providers_df['HCPCS_Diversity'] = np.where(
    providers_df['Total_Services_B'] > 0,
    providers_df['Unique_HCPCS_B'] / providers_df['Total_Services_B'],
    0
)

providers_df['Procedure_Breadth'] = providers_df['Unique_HCPCS_B']

providers_df['Total_Patient_Volume'] = providers_df['Total_Beneficiaries_B']

providers_df['Has_PartB'] = (providers_df['Total_Services_B'] > 0).astype(int)
providers_df['Has_PartD'] = (providers_df['Total_Claims_D']   > 0).astype(int)
providers_df['Has_Both']  = (
    (providers_df['Has_PartB'] == 1) & (providers_df['Has_PartD'] == 1)
).astype(int)


print("[3] Drug cost & prescribing pattern features ...")

providers_df['Drug_Cost_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Drug_Cost_D'] / providers_df['Total_Claims_D'],
    0
)

providers_df['Avg_Cost_per_Claim_D'] = providers_df['Drug_Cost_Per_Claim_D']
providers_df['Drug_Cost_Per_Day'] = np.where(
    providers_df['Total_Day_Supply_D'] > 0,
    providers_df['Total_Drug_Cost_D'] / providers_df['Total_Day_Supply_D'],
    0
)

providers_df['Fills_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_30Day_Fills_D'] / providers_df['Total_Claims_D'],
    0
)

providers_df['Days_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Day_Supply_D'] / providers_df['Total_Claims_D'],
    0
)

print("[4] Drug diversity & brand features ...")

providers_df['Brand_Ratio'] = np.where(
    (providers_df['Unique_Brand_Drugs_D'] + providers_df['Unique_Generic_Drugs_D']) > 0,
    providers_df['Unique_Brand_Drugs_D'] /
        (providers_df['Unique_Brand_Drugs_D'] + providers_df['Unique_Generic_Drugs_D']),
    0
)

providers_df['Total_Service_Diversity'] = (
    providers_df['Service_Type_Diversity_B'] +
    providers_df['Unique_Brand_Drugs_D'] +
    providers_df['Unique_Generic_Drugs_D']
)


print("[5] Cross-part B ↔ D features ...")

providers_df['Service_to_Drug_Ratio'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Services_B'] / providers_df['Total_Claims_D'],
    0
)

print("[6] Log-transform features (log1p, zero-safe) ...")

providers_df['log_Total_Services_B']      = np.log1p(providers_df['Total_Services_B'])
providers_df['log_Total_Beneficiaries_B'] = np.log1p(providers_df['Total_Beneficiaries_B'])
providers_df['log_Total_Claims_D']        = np.log1p(providers_df['Total_Claims_D'])
providers_df['log_Total_Drug_Cost_D']     = np.log1p(providers_df['Total_Drug_Cost_D'])
providers_df['log_Total_Day_Supply_D']    = np.log1p(providers_df['Total_Day_Supply_D'])

print("[7] Within-Provider_Type z-score features ...")

for raw_col, z_col in [
    ('Total_Services_B',      'Z_Total_Services_B'),
    ('Total_Drug_Cost_D',     'Z_Total_Drug_Cost_D'),
    ('Avg_Medicare_Payment_B','Z_Avg_Medicare_Payment_B'),
]:
    _grp  = providers_df.groupby('Provider_Type')[raw_col]
    _mean = _grp.transform('mean')
    _std  = _grp.transform('std').replace(0, np.nan)
    providers_df[z_col] = ((providers_df[raw_col] - _mean) / _std).fillna(0)

print("[8] Binary high-value anomaly flags (z > 1.645 = top-5% within specialty) ...")

providers_df['High_Service_Volume_Flag'] = (
    providers_df['Z_Total_Services_B']       > 1.645).astype(int)
providers_df['High_Drug_Cost_Flag']      = (
    providers_df['Z_Total_Drug_Cost_D']      > 1.645).astype(int)
providers_df['High_Payment_Flag']        = (
    providers_df['Z_Avg_Medicare_Payment_B'] > 1.645).astype(int)

print("[9] Interaction features ...")

providers_df['High_Service_High_Payment'] = (
    providers_df['Services_per_Beneficiary_B'] * providers_df['Avg_Medicare_Payment_B']
)

providers_df['Drug_Intensity'] = (
    providers_df['Drug_Cost_Per_Claim_D'] * providers_df['Brand_Ratio']
)
providers_df['Charge_x_Volume'] = (
    providers_df['Avg_Submitted_Charge_B'] * providers_df['Total_Services_B']
)

_n_new = providers_df.shape[1] - _n_cols_before

print(f"Final shape     : {providers_df.shape[0]:,} rows × {providers_df.shape[1]:,} columns")
print(f"Base columns    : {_n_cols_before}")
print(f"New features    : {_n_new}")
print(f"Missing values  : {providers_df.isnull().sum().sum():,}")
print("✓ Feature engineering complete!")

providers_df.head()


#### 7.1.1 Summary Statistics for Engineered Features

In [ ]:
print("7.1.1  Summary Statistics — Engineered Features")


engineered_features = {
    
    "--- GROUP 1: Payment & Charge Ratios ---": [],
    "Charge_to_Allowed_Ratio"      : "Over-billing rate vs. Medicare allowed (>1 = upcoding risk)",
    "Payment_to_Allowed_Ratio"     : "Medicare payment as fraction of allowed (legitimately ≤1)",
    "MaxCharge_Ratio"              : "Max vs. avg charge — outlier charge spike detector",
    "Allowed_Charge_Gap"           : "Absolute $ over-billing per service line",
    "Payment_Efficiency"           : "Fraction of submitted charge actually paid by Medicare",
    "Standardized_Payment_Ratio"   : "Payment premium above geography-adjusted baseline",

    
    "--- GROUP 2: Service Intensity (Part B) ---": [],
    "Services_per_Beneficiary_B"   : "Average services billed per unique patient",
    "BeneDay_Per_Beneficiary_B"    : "Beneficiary-day services per patient (phantom-billing flag)",
    "Services_Per_HCPCS"           : "Billing code concentration (repetitive-code flag)",
    "HCPCS_Diversity"              : "Code diversity ratio — low = concentrated billing",
    "Procedure_Breadth"            : "Unique HCPCS codes (named alias for Unique_HCPCS_B)",
    "Total_Patient_Volume"         : "Unique patient count (alias for Total_Beneficiaries_B)",
    "Has_PartB"                    : "Binary: provider has Part B activity",
    "Has_PartD"                    : "Binary: provider has Part D activity",
    "Has_Both"                     : "Binary: provider participates in both Part B and D",

   
    "--- GROUP 3: Drug Cost & Prescribing (Part D) ---": [],
    "Drug_Cost_Per_Claim_D"        : "Weighted avg drug cost per filed claim (sum÷sum)",
    "Avg_Cost_per_Claim_D"         : "Alias for Drug_Cost_Per_Claim_D (backward compat.)",
    "Drug_Cost_Per_Day"            : "Daily drug cost — high = expensive/high-dose prescribing",
    "Fills_Per_Claim_D"            : "30-day fills per claim — >1 flags drug-mill patterns",
    "Days_Per_Claim_D"             : "Avg days supply per claim — very low = frequent refills",

  
    "--- GROUP 4: Drug Diversity & Brand ---": [],
    "Brand_Ratio"                  : "Fraction of unique drugs that are brand-name (0–1)",
    "Total_Service_Diversity"      : "Breadth of services across Part B + Part D",

  
    "--- GROUP 5: Cross-Part (B ↔ D) ---": [],
    "Service_to_Drug_Ratio"        : "Part B services / Part D claims (specialty mismatch flag)",

    
    "--- GROUP 6: Log Transforms ---": [],
    "log_Total_Services_B"         : "log(1 + Total_Services_B) — right-skew compression",
    "log_Total_Beneficiaries_B"    : "log(1 + Total_Beneficiaries_B)",
    "log_Total_Claims_D"           : "log(1 + Total_Claims_D)",
    "log_Total_Drug_Cost_D"        : "log(1 + Total_Drug_Cost_D)",
    "log_Total_Day_Supply_D"       : "log(1 + Total_Day_Supply_D)",


    "--- GROUP 7: Within-Provider_Type Z-scores ---": [],
    "Z_Total_Services_B"           : "Peer-relative service volume (vs. same Provider_Type)",
    "Z_Total_Drug_Cost_D"          : "Peer-relative drug spend (vs. same Provider_Type)",
    "Z_Avg_Medicare_Payment_B"     : "Peer-relative payment per service (vs. same Provider_Type)",

    "--- GROUP 8: Binary Anomaly Flags (top-5% within specialty) ---": [],
    "High_Service_Volume_Flag"     : "1 if Z_Total_Services_B > 1.645",
    "High_Drug_Cost_Flag"          : "1 if Z_Total_Drug_Cost_D > 1.645",
    "High_Payment_Flag"            : "1 if Z_Avg_Medicare_Payment_B > 1.645",

    "--- GROUP 9: Interaction Features ---": [],
    "High_Service_High_Payment"    : "Services_per_Beneficiary_B × Avg_Medicare_Payment_B",
    "Drug_Intensity"               : "Drug_Cost_Per_Claim_D × Brand_Ratio",
    "Charge_x_Volume"              : "Avg_Submitted_Charge_B × Total_Services_B (total exposure)",
}

summary_data = []
for key, desc in engineered_features.items():
    if key.startswith("---"):
        print(f"\n{key}")
        print("-" * 80)
        continue
    if key not in providers_df.columns:
        print(f"  ⚠ MISSING: {key}")
        continue
    col = providers_df[key]
    fraud_mean    = providers_df.loc[providers_df['Is_Fraudulent'] == 1, key].mean()
    nonfr_mean    = providers_df.loc[providers_df['Is_Fraudulent'] == 0, key].mean()
    fraud_lift    = (fraud_mean / nonfr_mean) if nonfr_mean != 0 else float('nan')
    print(f"  {key:<32}  min={col.min():>10.3f}  max={col.max():>12.3f}  "
          f"mean={col.mean():>10.3f}  fraud_lift={fraud_lift:>6.2f}x  | {desc[:40]}")
    summary_data.append({
        'Feature'          : key,
        'Min'              : col.min(),
        'Max'              : col.max(),
        'Mean'             : col.mean(),
        'Median'           : col.median(),
        'Fraud Mean'       : fraud_mean,
        'Non-Fraud Mean'   : nonfr_mean,
        'Fraud Lift'       : fraud_lift,
        'Description'      : desc,
    })

summary_df = pd.DataFrame(summary_data)

print(f"✓ Total engineered features tracked : {len(summary_data)}")
print(f"✓ Successfully created              : {sum(1 for d in summary_data if d['Feature'] in providers_df.columns)}")
print(f"\nTop features by Fraud Lift (fraud mean / non-fraud mean):")
top_lift = (summary_df[summary_df['Fraud Lift'].notna() & np.isfinite(summary_df['Fraud Lift'])]
            .nlargest(10, 'Fraud Lift')[['Feature', 'Fraud Mean', 'Non-Fraud Mean', 'Fraud Lift']])
print(top_lift.to_string(index=False))


### 7.2

In [ ]:
fraud_by_state = providers_df[providers_df['Is_Fraudulent'] == 1]['State'].value_counts().head(15)

fraud_rate_by_state = (
    providers_df.groupby('State')['Is_Fraudulent'].agg(['sum', 'count'])
)
fraud_rate_by_state['fraud_rate'] = (fraud_rate_by_state['sum'] / fraud_rate_by_state['count'] * 100)
fraud_rate_by_state = fraud_rate_by_state.sort_values('fraud_rate', ascending=False).head(15)

fig = make_subplots(
    rows=1, cols=1,
    subplot_titles=('Top 15 States by Fraud Rate %',)
)

fig.add_trace(
    go.Bar(x=fraud_rate_by_state['fraud_rate'], y=fraud_rate_by_state.index, 
           orientation='h', name='Fraud Rate %', marker_color='red'),
    row=1, col=1
)

fig.update_layout(height=400, showlegend=False, title_text="Geographic Distribution of Healthcare Fraud")
fig.update_xaxes(title_text="Fraud Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="State", row=1, col=1)
fig.show()

print(f"\nTop states by fraud count:")
print(fraud_by_state)

print(f"\nTop states by fraud rate:")
print(fraud_rate_by_state[['sum', 'count', 'fraud_rate']].head(15))


### 7.3 Specialty Analysis

In [ ]:
print("Analyzing fraud patterns by provider specialty...")

fraud_rate_by_type = (
    providers_df.groupby('Provider_Type')['Is_Fraudulent'].agg(['sum', 'count'])
)
fraud_rate_by_type = fraud_rate_by_type[fraud_rate_by_type['count'] >= 100] 
fraud_rate_by_type['fraud_rate'] = (fraud_rate_by_type['sum'] / fraud_rate_by_type['count'] * 100)
fraud_rate_by_type = fraud_rate_by_type.sort_values('fraud_rate', ascending=False).head(15)

fig = go.Figure()

fig.add_trace(
    go.Bar(y=fraud_rate_by_type.index, x=fraud_rate_by_type['fraud_rate'], orientation='h',
           marker_color='darkred', text=fraud_rate_by_type['fraud_rate'].round(2),
           texttemplate='%{text}%', textposition='outside')
)

fig.update_layout(
    height=600, 
    title_text="Healthcare Fraud Rate by Provider Specialty (min 100 providers)",
    xaxis_title="Fraud Rate (%)",
    yaxis_title="Provider Type",
    showlegend=False
)

fig.show()

print(f"\nTop 15 specialties by fraud rate:")
print(fraud_rate_by_type[['sum', 'count', 'fraud_rate']])


### 7.4 Correlation Analysis

In [ ]:
print("Analyzing feature correlations...")

numeric_features = providers_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove('NPI') 


corr_matrix = providers_df[numeric_features].corr()


plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


fraud_correlations = corr_matrix['Is_Fraudulent'].sort_values(ascending=False)
print("\nTop 15 features correlated with fraud:")
print(fraud_correlations.head(16)[1:]) 

print("\nBottom 10 features (negative correlation with fraud):")
print(fraud_correlations.tail(10))

## 8. Statistical Feature Selection

### 8.1

In [ ]:
from sklearn.feature_selection import VarianceThreshold

print("SECTION 8: STATISTICAL FEATURE SELECTION")

print("NOTE: ALL FEATURES RETAINED - No feature removal for this analysis")

columns_to_drop_for_selection = ['NPI', 'Is_Fraudulent', 'Provider_Name', 'Provider_FirstName', 
                                   'State', 'Zipcode', 'Provider_Type', 'Entity_Code', 'Data_Source']
existing_cols_to_drop_selection = [col for col in columns_to_drop_for_selection if col in providers_df.columns]

print(f"\n  • Dropping {len(existing_cols_to_drop_selection)} non-predictive columns: {existing_cols_to_drop_selection}")

X_all = providers_df.drop(columns=existing_cols_to_drop_selection)
y_all = providers_df['Is_Fraudulent']

print(f"\n✓ Total predictive features: {X_all.shape[1]}")
print(f"\n✓ Total samples: {X_all.shape[0]:,}")
print(f"✓ Fraud cases: {y_all.sum():,} ({y_all.sum()/len(y_all)*100:.2f}%)")

print("STEP 0: TRAIN-TEST SPLIT (80-20)")


from sklearn.model_selection import train_test_split

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_all, y_all, 
    test_size=0.20, 
    random_state=RANDOM_STATE, 
    stratify=y_all
)

print(f"✓ Training set size: {X_train_full.shape[0]:,} samples")
print(f"✓ Test set size: {X_test_full.shape[0]:,} samples")
print(f"✓ Fraud in train set: {y_train_full.sum():,} ({y_train_full.sum()/len(y_train_full)*100:.2f}%)")
print(f"✓ Fraud in test set: {y_test_full.sum():,} ({y_test_full.sum()/len(y_test_full)*100:.2f}%)")

print("STEP 1: LOW-VARIANCE FEATURE REMOVAL")

print("NOTE: ALL FEATURES RETAINED - No variance threshold applied")

selector_var = VarianceThreshold(threshold=0.0)
X_train_var = selector_var.fit_transform(X_train_full)
X_test_var = selector_var.transform(X_test_full)

X_train_var = pd.DataFrame(X_train_var, columns=X_train_full.columns)
X_test_var = pd.DataFrame(X_test_var, columns=X_test_full.columns)

print(f"  • Variance threshold: 0.0 (keep all features)")
print(f"  • Features removed: 0")
print(f"✓ Features remaining: {X_train_var.shape[1]}")

print("STEP 2: HIGHLY CORRELATED FEATURE REMOVAL")

print("NOTE: ALL FEATURES RETAINED - No correlation threshold applied")

X_train_corr = X_train_var.copy()
X_test_corr = X_test_var.copy()

print(f"  • Correlation threshold: None")
print(f"  • Features removed: 0")
print(f"✓ Features remaining: {X_train_corr.shape[1]}")

print("STEP 3: STATISTICAL SIGNIFICANCE TESTING (CHI-SQUARE)")

print("NOTE: ALL FEATURES RETAINED - No statistical selection applied")

X_train_stats = X_train_corr.copy()
X_test_stats = X_test_corr.copy()

print(f"  • Selection method: None (all features kept)")
print(f"  • Features removed: 0")
print(f"✓ Features remaining: {X_train_stats.shape[1]}")

print("FINAL FEATURE SET (STATISTICAL SELECTION)")

print(f"✓ Training samples: {X_train_stats.shape[0]:,}")
print(f"✓ Test samples: {X_test_stats.shape[0]:,}")
print(f"✓ Total features retained: {X_train_stats.shape[1]}")
print(f"✓ Features removed in entire process: 0 (all features kept)")


print(f"\nFeature names (first 10): {list(X_train_stats.columns[:10])}")
print(f"Feature names (last 10): {list(X_train_stats.columns[-10:])}")


X_train_selected = X_train_stats
X_test_selected = X_test_stats


DATASETS_DIR = OUTPUT_DIR / 'datasets'
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

train_data_full = pd.concat([X_train_selected, y_train_full.reset_index(drop=True)], axis=1)
test_data_full = pd.concat([X_test_selected, y_test_full.reset_index(drop=True)], axis=1)


train_data_full.to_csv(DATASETS_DIR / 'train_set_reduced_all fea_SMOTE.csv', index=False)
test_data_full.to_csv(DATASETS_DIR / 'test_set_reduced_all fea_SMOTE.csv', index=False)

pd.DataFrame({'Feature': X_train_selected.columns}).to_csv(
    DATASETS_DIR / 'selected_features_statistical_all fea_SMOTE.csv', 
    index=False
)

X_train_selected.to_csv(DATASETS_DIR / 'X_train_reduced_all fea_SMOTE.csv', index=False)
X_test_selected.to_csv(DATASETS_DIR / 'X_test_reduced_all fea_SMOTE.csv', index=False)

print(f"\n✓ Datasets saved to: {DATASETS_DIR}")
print(f"  • train_set_reduced_all fea_SMOTE.csv")
print(f"  • test_set_reduced_all fea_SMOTE.csv")
print(f"  • selected_features_statistical_all fea_SMOTE.csv")
print(f"  • X_train_reduced_all fea_SMOTE.csv")
print(f"  • X_test_reduced_all fea_SMOTE.csv")

### 8.2

In [ ]:
export_dir = OUTPUT_DIR / 'datasets'
export_dir.mkdir(parents=True, exist_ok=True)

X_train_current = X_train_selected.copy()
X_test_current = X_test_selected.copy()
y_train_current = y_train_full.copy()
y_test_current = y_test_full.copy()


train_export = X_train_current.copy()
train_export['Is_Fraudulent'] = y_train_current.values

test_export = X_test_current.copy()
test_export['Is_Fraudulent'] = y_test_current.values

X_train_current.to_csv(export_dir / 'X_train_all_features_Without_Sampling.csv', index=True, index_label='original_index')
X_test_current.to_csv(export_dir / 'X_test_all_features_Without_Sampling.csv', index=True, index_label='original_index')
y_train_current.to_csv(export_dir / 'y_train_Without_Sampling.csv', index=True, index_label='original_index', header=True)
y_test_current.to_csv(export_dir / 'y_test_Without_Sampling.csv', index=True, index_label='original_index', header=True)

train_export.to_csv(export_dir / 'train_set_all_features_Without_Sampling.csv', index=True, index_label='original_index')
test_export.to_csv(export_dir / 'test_set_all_features_Without_Sampling.csv', index=True, index_label='original_index')


pd.Series(X_train_current.columns, name='feature').to_csv(
    export_dir / 'selected_features_all_features_Without_Sampling.csv', index=False
)

print("✓ Train/test sets exported with ALL features")
print(f"Export folder: {export_dir}")
print(f"Train features: {X_train_current.shape}")
print(f"Test features:  {X_test_current.shape}")
print(f"Train labels:   {y_train_current.shape}")
print(f"Test labels:    {y_test_current.shape}")
print("\nFiles created:")
for file_name in [
    'X_train_all_features_Without_Sampling.csv',
    'X_test_all_features_Without_Sampling.csv',
    'y_train_Without_Sampling.csv',
    'y_test_Without_Sampling.csv',
    'train_set_all_features_Without_Sampling.csv',
    'test_set_all_features_Without_Sampling.csv',
    'selected_features_all_features_Without_Sampling.csv'
]:
    print(f"  - {file_name}")

### 8.3

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path('/Users/roya/Desktop/Thesis/Thesis_Project/ML')

export_dir = OUTPUT_DIR / 'datasets'

X_train_reduced = pd.read_csv(export_dir / 'X_train_all_features_Without_Sampling.csv', index_col='original_index')
X_test_reduced = pd.read_csv(export_dir / 'X_test_all_features_Without_Sampling.csv', index_col='original_index')

y_train = pd.read_csv(export_dir / 'y_train_Without_Sampling.csv', index_col='original_index').iloc[:, 0]
y_test = pd.read_csv(export_dir / 'y_test_Without_Sampling.csv', index_col='original_index').iloc[:, 0]

selected_features_statistical = pd.read_csv(
    export_dir / 'selected_features_all_features_Without_Sampling.csv'
)['feature'].tolist()

train_export = pd.read_csv(export_dir / 'train_set_all_features_Without_Sampling.csv', index_col='original_index')
test_export = pd.read_csv(export_dir / 'test_set_all_features_Without_Sampling.csv', index_col='original_index')

print("✓ Exported files loaded successfully (ALL features)")
print(f"Folder: {export_dir}")
print(f"X_train_reduced (all features): {X_train_reduced.shape}")
print(f"X_test_reduced (all features):  {X_test_reduced.shape}")
print(f"y_train:                        {y_train.shape}")
print(f"y_test:                         {y_test.shape}")
print(f"Total features:                 {len(selected_features_statistical)}")

print("\nFirst 5 features:")
print(selected_features_statistical[:5])


## 9. Data Scaling (No Sampling — Class Imbalance Handled in Models)
### 9.1 Data Summary (Using Pre-Split Data)

In [ ]:
print("DATA PREPARATION SUMMARY")


print("\n📊 Using pre-split data from Section 8.1:")
print(f"   • Train-test split already performed (80-20 stratified)")
print(f"   • ALL features retained (no feature removal)")
print(f"   • No data leakage - test data never influenced analysis")

print(f"\n✓ Training set shape: {X_train_reduced.shape}")
print(f"✓ Test set shape: {X_test_reduced.shape}")

print(f"\n✓ Training fraud rate: {y_train.mean()*100:.4f}% ({y_train.sum():,} / {len(y_train):,})")
print(f"✓ Test fraud rate: {y_test.mean()*100:.4f}% ({y_test.sum():,} / {len(y_test):,})")

print(f"\n✓ Total features for modeling: {len(selected_features_statistical)}")
print("\nFeature list (all features):")
for i, feat in enumerate(selected_features_statistical, 1):
    print(f"  {i:2d}. {feat}")

X_train = X_train_reduced.copy()
X_test = X_test_reduced.copy()
feature_names = X_train.columns.tolist()

print(f"\n✓ Ready for modeling with {len(feature_names)} features (complete feature set)")


### 9.2 Class Imbalance Strategy (No Resampling)

In [ ]:
print("CLASS IMBALANCE STRATEGY: No resampling applied")


n_fraud    = int((y_train == 1).sum())
n_nonfraud = int((y_train == 0).sum())

print(f"\nTraining set — raw class distribution:")
print(f"  Class 0 (Non-Fraud): {n_nonfraud:,}")
print(f"  Class 1 (Fraud):     {n_fraud:,}")
print(f"  Imbalance ratio:     {n_nonfraud / n_fraud:.2f}:1")
print(f"  Fraud prevalence:    {n_fraud / (n_fraud + n_nonfraud) * 100:.4f}%")

X_train_balanced = X_train.copy()
y_train_balanced = y_train.copy()

print(f"\n✓ No resampling performed.")
print(f"  X_train_balanced shape : {X_train_balanced.shape}")
print(f"  y_train_balanced shape : {y_train_balanced.shape}")
print(f"\n→ Class imbalance will be handled in each model via:")
print(f"   • class_weight='balanced'               (LogReg, RF, SVM, HistGBM)")
print(f"   • class_weight='balanced' (is_unbalance) (LightGBM)")
print(f"   • scale_pos_weight={n_nonfraud/n_fraud:.1f}          (XGBoost)")
print(f"   • sample_weight via compute_sample_weight (MLP)")
print(f"   • CrossEntropyLoss(weight=...)           (GraphSAGE — already implemented)")


In [ ]:
print("No resampling applied — seed-stability probe not applicable.")
print(f"Training set: {X_train_balanced.shape[0]:,} rows  "
      f"({int(y_train_balanced.sum()):,} fraud, "
      f"{int((y_train_balanced == 0).sum()):,} non-fraud)")


### 9.3 Feature Scaling

In [ ]:
from sklearn.preprocessing import RobustScaler

print("Scaling features using RobustScaler...")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Training data scaled: {X_train_scaled.shape}")
print(f"✓ Test data scaled: {X_test_scaled.shape}")

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

print("\nData is ready for model training!")


### 9.4 Model Evaluation Function

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, classification_report
)

def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
   
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"MCC:       {mcc:.4f}")
  
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred))
    
    return {
        'model': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'mcc': mcc
    }

print("✓ Model evaluation function defined")


## 10. Baseline Models

In [ ]:

results_list = []

print("MODEL TRAINING AND EVALUATION")
print("\nInitialized results tracking...")
print(f"Test set size: {len(y_test):,} samples")
print(f"Test set fraud cases: {y_test.sum():,} ({y_test.sum()/len(y_test)*100:.2f}%)")

y_test.value_counts()

In [ ]:
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
print("Training Logistic Regression Model...")

lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train_balanced)

y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_results = evaluate_model(y_test, y_pred_lr, y_pred_proba_lr, "Logistic Regression")
results_list.append(lr_results)

print("\n✓ Logistic Regression training complete!")


## 11. Optimized Models


In [ ]:
# Random Forest 

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings


print("Section 11.1  Hyperparameter Tuning: Random Forest")
print(f"\nOriginal training samples (no resampling) : {len(y_train_balanced):,}")
print(f"Fraud cases                               : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")

X_tr_rf, X_val_rf, y_tr_rf, y_val_rf = train_test_split(
    X_train_scaled, y_train_balanced,
    test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
)
print(f"Train subset : {len(y_tr_rf):,} | Val subset : {len(y_val_rf):,}")
print(f"Val fraud    : {int(y_val_rf.sum()):,}")

cv_rf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

space_rf = {
    'n_estimators'     : scope.int(hp.quniform('n_estimators',      150, 450, 50)),
    'max_depth'        : hp.choice('max_depth',        [3, 4, 5, 6, 7, 8]),
    'min_samples_leaf' : scope.int(hp.quniform('min_samples_leaf',   4,  12,  2)),
    'min_samples_split': scope.int(hp.quniform('min_samples_split',  8,  18,  2)),
    'max_features'     : hp.choice('max_features',     [0.3, 0.4, 0.5, 0.6, 'sqrt']),
    'max_samples'      : hp.choice('max_samples',      [0.7, 0.8, 0.9, None]),
}

def _objective_rf(params):
    m = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced_subsample')
    scores = cross_val_score(m, X_tr_rf, y_tr_rf, cv=cv_rf, scoring='roc_auc', n_jobs=1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_rf = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_rf = fmin(
        fn=_objective_rf, space=space_rf, algo=tpe.suggest,
        max_evals=60, trials=trials_rf,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_rf_params = space_eval(space_rf, _best_rf)
print(f"\nBest RF params  : {best_rf_params}")
print(f"Best CV ROC-AUC : {-trials_rf.best_trial['result']['loss']:.4f}")

_rf_uncalib = RandomForestClassifier(**best_rf_params, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced_subsample')
_rf_uncalib.fit(X_tr_rf, y_tr_rf)
calibrated_rf = CalibratedClassifierCV(_rf_uncalib, cv='prefit', method='isotonic')
calibrated_rf.fit(X_val_rf, y_val_rf)

val_proba_rf    = calibrated_rf.predict_proba(X_val_rf)[:, 1]
rf_fraud_probas = val_proba_rf[y_val_rf == 1]
print(f"\nCalibrated fraud probabilities — min: {rf_fraud_probas.min():.4f}  "
      f"max: {rf_fraud_probas.max():.4f}  mean: {rf_fraud_probas.mean():.4f}")

fprs_rf, tprs_rf, roc_thr_rf = roc_curve(y_val_rf, val_proba_rf)
youden_rf   = tprs_rf - fprs_rf
best_idx_rf = int(np.argmax(youden_rf))
best_thr_rf = float(roc_thr_rf[best_idx_rf])
print(f"\nYouden's J threshold : {best_thr_rf:.4f}")
print(f"  Val TPR (recall)   : {tprs_rf[best_idx_rf]:.4f}")
print(f"  Val FPR            : {fprs_rf[best_idx_rf]:.4f}")

print(f"\nRefitting best params on full training set ({len(y_train_balanced):,} rows)…")
_rf_full = RandomForestClassifier(**best_rf_params, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced_subsample')
_rf_full.fit(X_train_scaled, y_train_balanced)
calibrated_rf_final = CalibratedClassifierCV(_rf_full, cv='prefit', method='isotonic')
calibrated_rf_final.fit(X_val_rf, y_val_rf)
best_rf_model = calibrated_rf_final

rf_tuned_proba   = best_rf_model.predict_proba(X_test_scaled)[:, 1]
rf_tuned_pred    = (rf_tuned_proba >= best_thr_rf).astype(int)
rf_tuned_results = evaluate_model(y_test, rf_tuned_pred, rf_tuned_proba, "Random Forest (Tuned)")
results_list = [r for r in results_list if r.get('model') != 'Random Forest (Tuned)']
results_list.append(rf_tuned_results)
print("\n✓ Random Forest tuning complete!")


In [ ]:
# XGBoost 


from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings

print("Section 11.2  Hyperparameter Tuning: XGBoost")

try:
    import xgboost as xgb

    print(f"\nOriginal training samples (no resampling) : {len(y_train_balanced):,}")
    print(f"Fraud cases                               : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")


    _neg_xgb = int((y_train_balanced == 0).sum())
    _pos_xgb = int((y_train_balanced == 1).sum())
    _spw     = _neg_xgb / _pos_xgb
    print(f"scale_pos_weight = {_spw:.2f}  ({_neg_xgb:,} non-fraud / {_pos_xgb:,} fraud)")

    X_tr_xgb, X_val_xgb, y_tr_xgb, y_val_xgb = train_test_split(
        X_train_scaled, y_train_balanced,
        test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
    )
    print(f"Train subset : {len(y_tr_xgb):,} | Val subset : {len(y_val_xgb):,}")
    print(f"Val fraud    : {int(y_val_xgb.sum()):,}")

    cv_xgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    space_xgb = {
        'n_estimators'    : scope.int(hp.quniform('n_estimators',      300, 600,  50)),
        'max_depth'       : scope.int(hp.quniform('max_depth',           2,   5,   1)),
        'learning_rate'   : hp.loguniform('learning_rate',   np.log(0.01),  np.log(0.10)),
        'subsample'       : hp.uniform('subsample',          0.5, 0.80),
        'colsample_bytree': hp.uniform('colsample_bytree',   0.70, 1.0),
        'min_child_weight': scope.int(hp.quniform('min_child_weight',   10,  25,   1)),
        'gamma'           : hp.loguniform('gamma',           np.log(0.10),  np.log(3.0)),
        'reg_alpha'       : hp.loguniform('reg_alpha',       np.log(0.50),  np.log(15.0)),
        'reg_lambda'      : hp.loguniform('reg_lambda',      np.log(0.50),  np.log(15.0)),
    }

    def _objective_xgb(params):
        m = xgb.XGBClassifier(
            **params,
            random_state=RANDOM_STATE, n_jobs=-1,
            tree_method='hist', eval_metric='logloss', verbosity=0,
            scale_pos_weight=_spw
        )
        scores = cross_val_score(m, X_tr_xgb, y_tr_xgb, cv=cv_xgb, scoring='roc_auc', n_jobs=1)
        return {'loss': -scores.mean(), 'status': STATUS_OK}

    trials_xgb = Trials()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _best_xgb = fmin(
            fn=_objective_xgb, space=space_xgb, algo=tpe.suggest,
            max_evals=60, trials=trials_xgb,
            rstate=np.random.default_rng(RANDOM_STATE), verbose=False
        )

    best_xgb_params = space_eval(space_xgb, _best_xgb)
    print(f"\nBest XGB params  : {best_xgb_params}")
    print(f"Best CV ROC-AUC  : {-trials_xgb.best_trial['result']['loss']:.4f}")

    _xgb_uncalib = xgb.XGBClassifier(
        **best_xgb_params,
        random_state=RANDOM_STATE, n_jobs=-1,
        tree_method='hist', eval_metric='logloss', verbosity=0,
        scale_pos_weight=_spw
    )
    _xgb_uncalib.fit(X_tr_xgb, y_tr_xgb)
    calibrated_xgb = CalibratedClassifierCV(_xgb_uncalib, cv='prefit', method='isotonic')
    calibrated_xgb.fit(X_val_xgb, y_val_xgb)

    val_proba_xgb    = calibrated_xgb.predict_proba(X_val_xgb)[:, 1]
    xgb_fraud_probas = val_proba_xgb[y_val_xgb == 1]
    print(f"\nCalibrated fraud probabilities — min: {xgb_fraud_probas.min():.4f}  "
          f"max: {xgb_fraud_probas.max():.4f}  mean: {xgb_fraud_probas.mean():.4f}")

    fprs_xgb, tprs_xgb, roc_thr_xgb = roc_curve(y_val_xgb, val_proba_xgb)
    youden_xgb   = tprs_xgb - fprs_xgb
    best_idx_xgb = int(np.argmax(youden_xgb))
    best_thr_xgb = float(roc_thr_xgb[best_idx_xgb])
    print(f"\nYouden's J threshold : {best_thr_xgb:.4f}")
    print(f"  Val TPR (recall)   : {tprs_xgb[best_idx_xgb]:.4f}")
    print(f"  Val FPR            : {fprs_xgb[best_idx_xgb]:.4f}")

    print(f"\nRefitting best params on full training set ({len(y_train_balanced):,} rows)…")
    _xgb_full = xgb.XGBClassifier(
        **best_xgb_params,
        random_state=RANDOM_STATE, n_jobs=-1,
        tree_method='hist', eval_metric='logloss', verbosity=0,
        scale_pos_weight=_spw
    )
    _xgb_full.fit(X_train_scaled, y_train_balanced)
    calibrated_xgb_final = CalibratedClassifierCV(_xgb_full, cv='prefit', method='isotonic')
    calibrated_xgb_final.fit(X_val_xgb, y_val_xgb)
    best_xgb_model = calibrated_xgb_final

    xgb_tuned_proba   = best_xgb_model.predict_proba(X_test_scaled)[:, 1]
    xgb_tuned_pred    = (xgb_tuned_proba >= best_thr_xgb).astype(int)
    xgb_tuned_results = evaluate_model(y_test, xgb_tuned_pred, xgb_tuned_proba, "XGBoost (Tuned)")
    results_list = [r for r in results_list if r.get('model') != 'XGBoost (Tuned)']
    results_list.append(xgb_tuned_results)
    print("\n✓ XGBoost tuning complete!")

except ImportError:
    print("⚠  XGBoost not available — skipping Section 11.2.")


In [ ]:
# LightGBM 

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from sklearn.model_selection import train_test_split, StratifiedKFold
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings

try:
    import lightgbm as lgb
    print("Section 11.3  Hyperparameter Tuning: LightGBM")


    print(f"\nOriginal training samples (no resampling) : {len(y_train_balanced):,}")
    print(f"Fraud cases                               : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")

    X_tr_lgb, X_val_lgb, y_tr_lgb, y_val_lgb = train_test_split(
        X_train_scaled, y_train_balanced,
        test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
    )
    print(f"Train subset : {len(y_tr_lgb):,} | Val subset : {len(y_val_lgb):,}")
    print(f"Val fraud    : {int(y_val_lgb.sum()):,}")

    cv_lgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    space_lgb = {
        'n_estimators'     : scope.int(hp.quniform('n_estimators',       100, 500, 50)),
        'max_depth'        : hp.choice('max_depth',      [4, 5, 6, 8, -1]),
        'learning_rate'    : hp.loguniform('learning_rate',  np.log(0.03), np.log(0.20)),
        'num_leaves'       : scope.int(hp.quniform('num_leaves',           15,  63,  4)),
        'subsample'        : hp.uniform('subsample',         0.70, 1.0),
        'colsample_bytree' : hp.uniform('colsample_bytree',  0.70, 1.0),
        'min_child_samples': scope.int(hp.quniform('min_child_samples',    50, 200, 10)),
        'lambda_l1'        : hp.loguniform('lambda_l1',  np.log(0.001), np.log(5.0)),
        'lambda_l2'        : hp.loguniform('lambda_l2',  np.log(2.0),   np.log(50.0)),
        'min_gain_to_split': hp.loguniform('min_gain_to_split', np.log(1e-4), np.log(0.5)),
    }

    def _objective_lgb(params):
        m = lgb.LGBMClassifier(
            **params, random_state=RANDOM_STATE, n_jobs=-1,
            bagging_freq=1, verbosity=-1, class_weight='balanced'
        )
        scores = cross_val_score(m, X_tr_lgb, y_tr_lgb, cv=cv_lgb, scoring='roc_auc', n_jobs=1)
        return {'loss': -scores.mean(), 'status': STATUS_OK}

    trials_lgb = Trials()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _best_lgb = fmin(
            fn=_objective_lgb, space=space_lgb, algo=tpe.suggest,
            max_evals=60, trials=trials_lgb,
            rstate=np.random.default_rng(RANDOM_STATE), verbose=False
        )

    best_lgb_params = space_eval(space_lgb, _best_lgb)
    print(f"\nBest LGB params  : {best_lgb_params}")
    print(f"Best CV ROC-AUC  : {-trials_lgb.best_trial['result']['loss']:.4f}")
    _lgb_uncalib = lgb.LGBMClassifier(
        **best_lgb_params, random_state=RANDOM_STATE, n_jobs=-1,
        bagging_freq=1, verbosity=-1, class_weight='balanced'
    )
    _lgb_uncalib.fit(X_tr_lgb, y_tr_lgb)
    calibrated_lgb = CalibratedClassifierCV(_lgb_uncalib, cv='prefit', method='isotonic')
    calibrated_lgb.fit(X_val_lgb, y_val_lgb)

    val_proba_lgb    = calibrated_lgb.predict_proba(X_val_lgb)[:, 1]
    lgb_fraud_probas = val_proba_lgb[y_val_lgb == 1]
    print(f"\nCalibrated fraud probabilities — min: {lgb_fraud_probas.min():.4f}  "
          f"max: {lgb_fraud_probas.max():.4f}  mean: {lgb_fraud_probas.mean():.4f}")
    fprs_lgb, tprs_lgb, roc_thr_lgb = roc_curve(y_val_lgb, val_proba_lgb)
    youden_lgb   = tprs_lgb - fprs_lgb
    best_idx_lgb = int(np.argmax(youden_lgb))
    best_thr_lgb = float(roc_thr_lgb[best_idx_lgb])
    print(f"\nYouden's J threshold : {best_thr_lgb:.4f}")
    print(f"  Val TPR (recall)   : {tprs_lgb[best_idx_lgb]:.4f}")
    print(f"  Val FPR            : {fprs_lgb[best_idx_lgb]:.4f}")

    print(f"\nRefitting best params on full training set ({len(y_train_balanced):,} rows)…")
    _lgb_full = lgb.LGBMClassifier(
        **best_lgb_params, random_state=RANDOM_STATE, n_jobs=-1,
        bagging_freq=1, verbosity=-1, class_weight='balanced'
    )
    _lgb_full.fit(X_train_scaled, y_train_balanced)
    calibrated_lgb_final = CalibratedClassifierCV(_lgb_full, cv='prefit', method='isotonic')
    calibrated_lgb_final.fit(X_val_lgb, y_val_lgb)
    best_lgb_model = calibrated_lgb_final

    lgb_tuned_proba   = best_lgb_model.predict_proba(X_test_scaled)[:, 1]
    lgb_tuned_pred    = (lgb_tuned_proba >= best_thr_lgb).astype(int)
    lgb_tuned_results = evaluate_model(y_test, lgb_tuned_pred, lgb_tuned_proba, "LightGBM (Tuned)")
    results_list = [r for r in results_list if r.get('model') != 'LightGBM (Tuned)']
    results_list.append(lgb_tuned_results)
    print("\n✓ LightGBM tuning complete!")

except ImportError:
    print("⚠  LightGBM not available — skipping Section 11.3.")


In [ ]:
# HistGradientBoostingClassifier 

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings


print("Section 11.4  Hyperparameter Tuning: HistGradientBoosting")
print(f"\nOriginal training samples (no resampling) : {len(y_train_balanced):,}")
print(f"Fraud cases                               : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")


X_tr_hgb, X_val_hgb, y_tr_hgb, y_val_hgb = train_test_split(
    X_train_scaled, y_train_balanced,
    test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
)
print(f"Train subset : {len(y_tr_hgb):,} | Val subset : {len(y_val_hgb):,}")
print(f"Val fraud    : {int(y_val_hgb.sum()):,}")

cv_hgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

space_hgb = {
    'max_iter'         : scope.int(hp.quniform('max_iter',          250, 550,  50)),
    'max_depth'        : hp.choice('max_depth',      [10, 12, 15, 20, None]),
    'learning_rate'    : hp.loguniform('learning_rate',  np.log(0.005), np.log(0.05)),
    'max_leaf_nodes'   : scope.int(hp.quniform('max_leaf_nodes',      15,  40,   5)),
    'min_samples_leaf' : scope.int(hp.quniform('min_samples_leaf',    60, 120,   5)),
    'l2_regularization': hp.loguniform('l2_regularization', np.log(0.50), np.log(20.0)),
    'max_bins'         : hp.choice('max_bins', [64, 128]),
}

def _objective_hgb(params):
    m = HistGradientBoostingClassifier(
        **params, random_state=RANDOM_STATE,
        early_stopping=False, verbose=0, class_weight='balanced'
    )
    scores = cross_val_score(m, X_tr_hgb, y_tr_hgb, cv=cv_hgb, scoring='roc_auc', n_jobs=-1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_hgb = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_hgb = fmin(
        fn=_objective_hgb, space=space_hgb, algo=tpe.suggest,
        max_evals=60, trials=trials_hgb,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_hgb_params = space_eval(space_hgb, _best_hgb)
print(f"\nBest HGB params  : {best_hgb_params}")
print(f"Best CV ROC-AUC  : {-trials_hgb.best_trial['result']['loss']:.4f}")

_hgb_uncalib = HistGradientBoostingClassifier(
    **best_hgb_params, random_state=RANDOM_STATE,
    early_stopping=False, verbose=0, class_weight='balanced'
)
_hgb_uncalib.fit(X_tr_hgb, y_tr_hgb)
calibrated_hgb = CalibratedClassifierCV(_hgb_uncalib, cv='prefit', method='isotonic')
calibrated_hgb.fit(X_val_hgb, y_val_hgb)

val_proba_hgb    = calibrated_hgb.predict_proba(X_val_hgb)[:, 1]
hgb_fraud_probas = val_proba_hgb[y_val_hgb == 1]
print(f"\nCalibrated fraud probabilities — min: {hgb_fraud_probas.min():.4f}  "
      f"max: {hgb_fraud_probas.max():.4f}  mean: {hgb_fraud_probas.mean():.4f}")

fprs_hgb, tprs_hgb, roc_thr_hgb = roc_curve(y_val_hgb, val_proba_hgb)
youden_hgb   = tprs_hgb - fprs_hgb
best_idx_hgb = int(np.argmax(youden_hgb))
best_thr_hgb = float(roc_thr_hgb[best_idx_hgb])
print(f"\nYouden's J threshold : {best_thr_hgb:.4f}")
print(f"  Val TPR (recall)   : {tprs_hgb[best_idx_hgb]:.4f}")
print(f"  Val FPR            : {fprs_hgb[best_idx_hgb]:.4f}")

print(f"\nRefitting best params on full training set ({len(y_train_balanced):,} rows)…")
_hgb_full = HistGradientBoostingClassifier(
    **best_hgb_params, random_state=RANDOM_STATE,
    early_stopping=False, verbose=0, class_weight='balanced'
)
_hgb_full.fit(X_train_scaled, y_train_balanced)
calibrated_hgb_final = CalibratedClassifierCV(_hgb_full, cv='prefit', method='isotonic')
calibrated_hgb_final.fit(X_val_hgb, y_val_hgb)
best_hgb_model = calibrated_hgb_final

hgb_tuned_proba   = best_hgb_model.predict_proba(X_test_scaled)[:, 1]
hgb_tuned_pred    = (hgb_tuned_proba >= best_thr_hgb).astype(int)
hgb_tuned_results = evaluate_model(
    y_test, hgb_tuned_pred, hgb_tuned_proba, "HistGradBoost (Tuned)"
)
results_list = [r for r in results_list if r.get('model') != 'HistGradBoost (Tuned)']
results_list.append(hgb_tuned_results)
print("\n✓ HistGradientBoosting tuning complete!")


In [ ]:
#  SVM 

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings


print("Section 11.5  Hyperparameter Tuning: SVM (LinearSVC)")
print(f"\nOriginal training samples (no resampling) : {len(y_train_balanced):,}")
print(f"Fraud cases                               : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")


X_tr_svm, X_val_svm, y_tr_svm, y_val_svm = train_test_split(
    X_train_scaled, y_train_balanced,
    test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
)
print(f"Train subset : {len(y_tr_svm):,} | Val subset : {len(y_val_svm):,}")
print(f"Val fraud    : {int(y_val_svm.sum()):,}")

cv_svm = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

space_svm = {
     'C': hp.loguniform('C', np.log(1e-3), np.log(5e-2)),
}

def _objective_svm(params):
    m = LinearSVC(
        C=params['C'], loss='squared_hinge',
        max_iter=5000, random_state=RANDOM_STATE, class_weight='balanced'
    )
    scores = cross_val_score(m, X_tr_svm, y_tr_svm, cv=cv_svm, scoring='roc_auc', n_jobs=-1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_svm = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_svm = fmin(
        fn=_objective_svm, space=space_svm, algo=tpe.suggest,
        max_evals=60, trials=trials_svm,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_svm_params = space_eval(space_svm, _best_svm)
print(f"\nBest SVM params  : {best_svm_params}")
print(f"Best CV ROC-AUC  : {-trials_svm.best_trial['result']['loss']:.4f}")

_svm_uncalib = LinearSVC(
    C=best_svm_params['C'], loss='squared_hinge',
    max_iter=5000, random_state=RANDOM_STATE, class_weight='balanced'
)
_svm_uncalib.fit(X_tr_svm, y_tr_svm)
calibrated_svm = CalibratedClassifierCV(_svm_uncalib, cv='prefit', method='sigmoid')
calibrated_svm.fit(X_val_svm, y_val_svm)

val_proba_svm    = calibrated_svm.predict_proba(X_val_svm)[:, 1]
svm_fraud_probas = val_proba_svm[y_val_svm == 1]
print(f"\nCalibrated fraud probabilities — min: {svm_fraud_probas.min():.4f}  "
      f"max: {svm_fraud_probas.max():.4f}  mean: {svm_fraud_probas.mean():.4f}")

fprs_svm, tprs_svm, roc_thr_svm = roc_curve(y_val_svm, val_proba_svm)
youden_svm   = tprs_svm - fprs_svm
best_idx_svm = int(np.argmax(youden_svm))
best_thr_svm = float(roc_thr_svm[best_idx_svm])
print(f"\nYouden's J threshold : {best_thr_svm:.4f}")
print(f"  Val TPR (recall)   : {tprs_svm[best_idx_svm]:.4f}")
print(f"  Val FPR            : {fprs_svm[best_idx_svm]:.4f}")

print(f"\nRefitting best params on full training set ({len(y_train_balanced):,} rows)…")
_svm_full = LinearSVC(
    C=best_svm_params['C'], loss='squared_hinge',
    max_iter=5000, random_state=RANDOM_STATE, class_weight='balanced'
)
_svm_full.fit(X_train_scaled, y_train_balanced)
calibrated_svm_final = CalibratedClassifierCV(_svm_full, cv='prefit', method='sigmoid')
calibrated_svm_final.fit(X_val_svm, y_val_svm)
best_svm_model = calibrated_svm_final

svm_tuned_proba   = best_svm_model.predict_proba(X_test_scaled)[:, 1]
svm_tuned_pred    = (svm_tuned_proba >= best_thr_svm).astype(int)
svm_tuned_results = evaluate_model(y_test, svm_tuned_pred, svm_tuned_proba, "SVM (Tuned)")
results_list = [r for r in results_list if r.get('model') != 'SVM (Tuned)']
results_list.append(svm_tuned_results)
print("\n✓ SVM tuning complete!")


In [ ]:
# MLPClassifier

from sklearn.neural_network import MLPClassifier
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings

print("Section 11.6  Hyperparameter Tuning: Neural Network (MLP)")
print(f"\nOriginal training set (no resampling) : {len(y_train_balanced):,} samples")
print(f"  Class 0 : {int((y_train_balanced==0).sum()):,}  |  "
      f"Class 1 : {int((y_train_balanced==1).sum()):,}")
print(f"Full training samples : {len(y_train):,}  (original)")
print(f"Fraud cases           : {int(y_train.sum()):,}  ({y_train.mean():.4%})")

if 'X_train_orig_scaled' not in dir():
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )
X_tr_mlp, X_val_mlp, y_tr_mlp_orig, y_val_mlp = train_test_split(
    X_train_orig_scaled, y_train,
    test_size=0.20, random_state=RANDOM_STATE + 5, stratify=y_train
)
print(f"\nCalibration val subset : {len(y_tr_mlp_orig):,} | {len(y_val_mlp):,}")
print(f"Val fraud              : {int(y_val_mlp.sum()):,}")

_MLP_CV_CAP = 20_000
if len(y_train_balanced) > _MLP_CV_CAP:
    _rng = np.random.RandomState(RANDOM_STATE)
    _idx = _rng.choice(len(y_train_balanced), size=_MLP_CV_CAP, replace=False)
    if hasattr(X_train_scaled, 'iloc'):
        X_mlp_cv = X_train_scaled.iloc[_idx]
        y_mlp_cv = y_train_balanced.iloc[_idx]
    else:
        X_mlp_cv = X_train_scaled[_idx]
        y_mlp_cv = y_train_balanced[_idx]
else:
    X_mlp_cv = X_train_scaled
    y_mlp_cv = y_train_balanced

print(f"\n⚡ CV search subsample : {len(y_mlp_cv):,} rows  "
      f"(fraud: {int((y_mlp_cv==1).sum()):,}, {(y_mlp_cv==1).mean():.4%})")

cv_mlp = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

space_mlp = {
    'hidden_layer_sizes': hp.choice('hidden_layer_sizes', [
        (128, 64),
        (256, 128, 64),
        (128, 64, 32),
        (128, 128),
        (256, 128),
        (64, 64, 32),
    ]),
    'alpha'             : hp.loguniform('alpha',              np.log(1e-5), np.log(1e-2)),
    'learning_rate_init': hp.loguniform('learning_rate_init', np.log(1e-4), np.log(2e-3)),
    'batch_size'        : hp.choice('batch_size', [128, 256, 512]),
}

def _objective_mlp(params):
    m = MLPClassifier(
        **params,
        activation='relu', solver='adam',
        learning_rate='adaptive',
        max_iter=300, random_state=RANDOM_STATE,
        early_stopping=True, validation_fraction=0.10,
        n_iter_no_change=15, verbose=False,
    )
    scores = cross_val_score(m, X_mlp_cv, y_mlp_cv, cv=cv_mlp, scoring='roc_auc', n_jobs=1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_mlp = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_mlp = fmin(
        fn=_objective_mlp, space=space_mlp, algo=tpe.suggest,
        max_evals=40, trials=trials_mlp,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_mlp_params = space_eval(space_mlp, _best_mlp)
print(f"\nBest MLP params  : {best_mlp_params}")
print(f"Best CV ROC-AUC  : {-trials_mlp.best_trial['result']['loss']:.4f}")

_mlp_uncalib = MLPClassifier(
    **best_mlp_params,
    activation='relu', solver='adam',
    learning_rate='adaptive',
    max_iter=300, random_state=RANDOM_STATE,
    early_stopping=True, validation_fraction=0.10,
    n_iter_no_change=15, verbose=False,
)
_mlp_uncalib.fit(X_train_scaled, y_train_balanced)
calibrated_mlp = CalibratedClassifierCV(_mlp_uncalib, cv='prefit', method='sigmoid')
calibrated_mlp.fit(X_val_mlp, y_val_mlp)

val_proba_mlp    = calibrated_mlp.predict_proba(X_val_mlp)[:, 1]
mlp_fraud_probas = val_proba_mlp[y_val_mlp == 1]
print(f"\nCalibrated fraud probabilities — min: {mlp_fraud_probas.min():.4f}  "
      f"max: {mlp_fraud_probas.max():.4f}  mean: {mlp_fraud_probas.mean():.4f}")

fprs_mlp, tprs_mlp, roc_thr_mlp = roc_curve(y_val_mlp, val_proba_mlp)
youden_mlp   = tprs_mlp - fprs_mlp
best_idx_mlp = int(np.argmax(youden_mlp))
best_thr_mlp = float(roc_thr_mlp[best_idx_mlp])
print(f"\nYouden's J threshold : {best_thr_mlp:.4f}")
print(f"  Val TPR (recall)   : {tprs_mlp[best_idx_mlp]:.4f}")
print(f"  Val FPR            : {fprs_mlp[best_idx_mlp]:.4f}")

print(f"\nRefitting best params on full training set ({len(y_train_balanced):,} rows)…")
_mlp_full = MLPClassifier(
    **best_mlp_params,
    activation='relu', solver='adam',
    learning_rate='adaptive',
    max_iter=500, random_state=RANDOM_STATE,
    early_stopping=True, validation_fraction=0.10,
    n_iter_no_change=20, verbose=False,
)
_mlp_full.fit(X_train_scaled, y_train_balanced)
calibrated_mlp_final = CalibratedClassifierCV(_mlp_full, cv='prefit', method='sigmoid')
calibrated_mlp_final.fit(X_val_mlp, y_val_mlp)
best_mlp_model = calibrated_mlp_final

mlp_tuned_proba   = best_mlp_model.predict_proba(X_test_scaled)[:, 1]
mlp_tuned_pred    = (mlp_tuned_proba >= best_thr_mlp).astype(int)
mlp_tuned_results = evaluate_model(y_test, mlp_tuned_pred, mlp_tuned_proba, "MLP (Tuned)")
results_list = [r for r in results_list if r.get('model') != 'MLP (Tuned)']
results_list.append(mlp_tuned_results)
print("\n✓ MLP tuning complete!")


### 11.7 Graph Neural Network — GraphSAGE  

In [ ]:
# Graph Neural Network
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from sklearn.linear_model import LogisticRegression as _CalLR
from sklearn.metrics import roc_curve

class _Adam:
    """Pure-Python Adam optimizer; sidesteps torch._dynamo import in torch.optim."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        self.params = [p for p in params if p.requires_grad]
        self.lr = lr; self.b1, self.b2 = betas
        self.eps = eps; self.wd = weight_decay; self.t = 0
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()

    def step(self):
        self.t += 1
        bc1, bc2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad.data
            if self.wd:
                g = g.add(p.data, alpha=self.wd)
            self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
            self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)

print("Section 14.1g  Graph Neural Network (GraphSAGE)")
print(f"  PyTorch {torch.__version__}  (pure-torch, no torch_geometric needed)")

_dev = (torch.device("mps")  if torch.backends.mps.is_available() else
        torch.device("cuda") if torch.cuda.is_available() else
        torch.device("cpu"))
print(f"  Device : {_dev}")


X_train_orig_scaled = scaler.transform(X_train)
X_train_orig_scaled = pd.DataFrame(X_train_orig_scaled, columns=feature_names)
print(f"  Created X_train_orig_scaled: {X_train_orig_scaled.shape}")

def _np32(s): return (s.values if hasattr(s,'values') else np.array(s)).astype('float32')
def _np64(s): return (s.values if hasattr(s,'values') else np.array(s)).astype('int64')

_X_tr_full = _np32(X_train_scaled);   _y_tr_full = _np64(y_train_balanced)
_X_te      = _np32(X_test_scaled);    _y_te      = _np64(y_test)
_Xo        = _np32(X_train_orig_scaled); _yo = _np64(y_train)

print(f"\nFull training set : {len(_y_tr_full):,}  "
      f"(fraud: {(_y_tr_full==1).sum():,} | non-fraud: {(_y_tr_full==0).sum():,})")
print(f"Test nodes        : {len(_y_te):,}  (fraud: {(_y_te==1).sum()})")

_SAGE_CAP = 30_000  
if len(_y_tr_full) > _SAGE_CAP:
    _rng_s   = np.random.RandomState(RANDOM_STATE + 3)
    _fr_idx  = np.where(_y_tr_full == 1)[0]
    _nf_idx  = np.where(_y_tr_full == 0)[0]
   
    _n_fr    = min(len(_fr_idx), _SAGE_CAP // 2)
    _fr_keep = (_rng_s.choice(_fr_idx, _n_fr, replace=False)
                if len(_fr_idx) > _n_fr else _fr_idx)
    _nf_keep = _rng_s.choice(_nf_idx, _SAGE_CAP - len(_fr_keep), replace=False)
    _keep    = np.concatenate([_fr_keep, _nf_keep])
    _keep.sort()
    _X_tr_full = _X_tr_full[_keep]
    _y_tr_full = _y_tr_full[_keep]
    print(f"Capped train nodes to {len(_keep):,}  "
          f"(fraud: {(_y_tr_full==1).sum()} | non-fraud: {(_y_tr_full==0).sum()})")

_N_tr, _d = _X_tr_full.shape
_N_te     = len(_y_te)

print("\nBuilding k-NN graph (k=10)...")
_k = 10
_knn = NearestNeighbors(n_neighbors=_k + 1, metric='euclidean', n_jobs=-1)
_knn.fit(_X_tr_full)
_dist, _idx = _knn.kneighbors(_X_tr_full, return_distance=True)

_edge_list = []
for node in range(_N_tr):
    for nei in _idx[node, 1:]:   # skip first (self)
        _edge_list.append((node, nei))
_edge_index = torch.tensor(_edge_list, dtype=torch.long, device=_dev).t().contiguous()
print(f"  Graph : {_N_tr} nodes, {len(_edge_list)} edges")

_X_tr_t = torch.tensor(_X_tr_full, dtype=torch.float32, device=_dev)
_y_tr_t = torch.tensor(_y_tr_full, dtype=torch.long,   device=_dev)
_X_te_t = torch.tensor(_X_te,      dtype=torch.float32, device=_dev)
_y_te_t = torch.tensor(_y_te,      dtype=torch.long,   device=_dev)

class _SAGEConv(nn.Module):
    """Single GraphSAGE convolution layer (mean aggregation)."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim, bias=False)
        self.lin_neig = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, edge_index):
        src, dst = edge_index
        N = x.size(0)
        neigh_sum = torch.zeros_like(x, device=x.device)
        neigh_cnt = torch.zeros(N, dtype=torch.float32, device=x.device)
        neigh_sum.index_add_(0, dst, x[src])
        neigh_cnt.index_add_(0, dst, torch.ones(src.size(0), device=x.device))
        neigh_cnt = neigh_cnt.clamp(min=1)  
        neigh_mean = neigh_sum / neigh_cnt.unsqueeze(1)
        out = self.lin_self(x) + self.lin_neig(neigh_mean)
        return out

class _GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.conv1 = _SAGEConv(in_dim, hidden)
        self.conv2 = _SAGEConv(hidden, hidden)
        self.fc    = nn.Linear(hidden, 1)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.fc(h).squeeze(-1)

_model_sage = _GraphSAGE(in_dim=_d, hidden=64).to(_dev)
_opt_sage   = _Adam(_model_sage.parameters(), lr=1e-3, weight_decay=1e-5)

print(f"\nModel : GraphSAGE(in={_d}, hidden=64, out=1)")
print(f"  Total parameters : {sum(p.numel() for p in _model_sage.parameters()):,}")

_EPOCHS_SAGE = 50
_model_sage.train()
print(f"\nTraining GraphSAGE for {_EPOCHS_SAGE} epochs ...")

for ep in range(_EPOCHS_SAGE):
    _opt_sage.zero_grad()
    _logits = _model_sage(_X_tr_t, _edge_index)
    _loss   = F.binary_cross_entropy_with_logits(_logits, _y_tr_t.float())
    _loss.backward()
    _opt_sage.step()

    if (ep + 1) % 10 == 0 or ep == 0:
        _probs_tr = torch.sigmoid(_logits).detach().cpu().numpy()
        _preds_tr = (_probs_tr >= 0.5).astype(int)
        _acc_tr   = (_preds_tr == _y_tr_full).mean()
        _rec_tr   = ((_preds_tr == 1) & (_y_tr_full == 1)).sum() / max((_y_tr_full == 1).sum(), 1)
        print(f"  Epoch {ep+1:3d} | Loss={_loss.item():.4f} | TrainAcc={_acc_tr:.4f} | TrainRecall={_rec_tr:.4f}")

print("\nTraining complete.")

_model_sage.eval()
with torch.no_grad():
    _edge_test = torch.empty((2, 0), dtype=torch.long, device=_dev)
    _logits_te = _model_sage(_X_te_t, _edge_test)
    _probs_te  = torch.sigmoid(_logits_te).cpu().numpy()
    _preds_te  = (_probs_te >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

_acc_te  = accuracy_score(_y_te, _preds_te)
_prec_te = precision_score(_y_te, _preds_te, zero_division=0)
_rec_te  = recall_score(_y_te, _preds_te, zero_division=0)
_f1_te   = f1_score(_y_te, _preds_te, zero_division=0)
_auc_te  = roc_auc_score(_y_te, _probs_te) if len(np.unique(_y_te)) > 1 else 0.0
_cm_te   = confusion_matrix(_y_te, _preds_te)


print("GraphSAGE Test Results")
print(f"  Accuracy  : {_acc_te:.4f}")
print(f"  Precision : {_prec_te:.4f}")
print(f"  Recall    : {_rec_te:.4f}")
print(f"  F1-Score  : {_f1_te:.4f}")
print(f"  AUC-ROC   : {_auc_te:.4f}")
print(f"\nConfusion Matrix:\n{_cm_te}")

print("Calibrating on original (unbalanced) training data...")

_Xo_t = torch.tensor(_Xo, dtype=torch.float32, device=_dev)
with torch.no_grad():
    _edge_orig = torch.empty((2, 0), dtype=torch.long, device=_dev)  # no graph for calibration
    _logits_o = _model_sage(_Xo_t, _edge_orig)
    _raw_scores_o = torch.sigmoid(_logits_o).cpu().numpy().ravel()

_cal_lr = _CalLR(max_iter=500, random_state=RANDOM_STATE)
_cal_lr.fit(_raw_scores_o.reshape(-1, 1), _yo)
_calibrated_probs_te = _cal_lr.predict_proba(_probs_te.reshape(-1, 1))[:, 1]

_fpr_cal, _tpr_cal, _thr_cal = roc_curve(_y_te, _calibrated_probs_te)
from sklearn.metrics import auc as sk_auc
_auc_cal = sk_auc(_fpr_cal, _tpr_cal)

_j_scores = _tpr_cal - _fpr_cal
_best_idx = np.argmax(_j_scores)
_best_thr = _thr_cal[_best_idx]
_preds_cal = (_calibrated_probs_te >= _best_thr).astype(int)

_acc_cal  = accuracy_score(_y_te, _preds_cal)
_prec_cal = precision_score(_y_te, _preds_cal, zero_division=0)
_rec_cal  = recall_score(_y_te, _preds_cal, zero_division=0)
_f1_cal   = f1_score(_y_te, _preds_cal, zero_division=0)
_cm_cal   = confusion_matrix(_y_te, _preds_cal)

print("\nCalibrated Results (best threshold = {:.4f}):".format(_best_thr))
print(f"  Accuracy  : {_acc_cal:.4f}")
print(f"  Precision : {_prec_cal:.4f}")
print(f"  Recall    : {_rec_cal:.4f}")
print(f"  F1-Score  : {_f1_cal:.4f}")
print(f"  AUC-ROC   : {_auc_cal:.4f}")
print(f"\nConfusion Matrix:\n{_cm_cal}")

print("\n✓ GraphSAGE training and evaluation complete.")

In [ ]:
# FNN

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression as _CalLR
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.metrics import auc as sk_auc

def _np32(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('float32')
def _np64(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('int64')

class _Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        self.params = [p for p in params if p.requires_grad]
        self.lr = lr; self.b1, self.b2 = betas
        self.eps = eps; self.wd = weight_decay; self.t = 0
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_(); p.grad.zero_()
    def step(self):
        self.t += 1
        bc1, bc2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t
        for i, p in enumerate(self.params):
            if p.grad is None: continue
            g = p.grad.data
            if self.wd: g = g.add(p.data, alpha=self.wd)
            self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
            self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            p.data.addcdiv_(self.m[i] / bc1, (self.v[i] / bc2).sqrt().add_(self.eps), value=-self.lr)

RANDOM_STATE = 42

if 'X_train_orig_scaled' not in dir() or X_train_orig_scaled is None:
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )

print("Section 14.1h  Feedforward Neural Network (FNN)")
print(f"  PyTorch {torch.__version__}")

_dev_fnn = (torch.device("mps")  if torch.backends.mps.is_available() else
            torch.device("cuda") if torch.cuda.is_available() else
            torch.device("cpu"))
print(f"  Device : {_dev_fnn}")

_FNN_CAP = 30_000
_X_tr_full_np = _np32(X_train_scaled)
_y_tr_full_np = _np64(y_train_balanced)

if len(_y_tr_full_np) > _FNN_CAP:
    _rng_fnn  = np.random.RandomState(RANDOM_STATE + 4)
    _fr_idx_f = np.where(_y_tr_full_np == 1)[0]
    _nf_idx_f = np.where(_y_tr_full_np == 0)[0]
    _n_fr_f   = min(len(_fr_idx_f), _FNN_CAP // 2)
    _fr_kp_f  = (_rng_fnn.choice(_fr_idx_f, _n_fr_f, replace=False)
                 if len(_fr_idx_f) > _n_fr_f else _fr_idx_f)
    _nf_kp_f  = _rng_fnn.choice(_nf_idx_f, _FNN_CAP - len(_fr_kp_f), replace=False)
    _keep_f   = np.sort(np.concatenate([_fr_kp_f, _nf_kp_f]))
    _X_tr_full_np = _X_tr_full_np[_keep_f]
    _y_tr_full_np = _y_tr_full_np[_keep_f]

_X_tr_fnn = torch.tensor(_X_tr_full_np,               dtype=torch.float32, device=_dev_fnn)
_y_tr_fnn = torch.tensor(_y_tr_full_np,               dtype=torch.long,    device=_dev_fnn)
_X_te_fnn = torch.tensor(_np32(X_test_scaled),        dtype=torch.float32, device=_dev_fnn)
_y_te_fnn = torch.tensor(_np64(y_test),               dtype=torch.long,    device=_dev_fnn)
_Xo_fnn   = torch.tensor(_np32(X_train_orig_scaled),  dtype=torch.float32, device=_dev_fnn)
_yo_fnn   = _np64(y_train)

_N_tr_fnn, _d_fnn = _X_tr_fnn.shape
_N_te_fnn = _X_te_fnn.shape[0]

print(f"\nTraining set (capped) : {_N_tr_fnn:,}  "
      f"(fraud: {(_y_tr_fnn==1).sum().item():,} | non-fraud: {(_y_tr_fnn==0).sum().item():,})")
print(f"Test set              : {_N_te_fnn:,}  (fraud: {(_y_te_fnn==1).sum().item():,})")

_n_fraud_fnn    = (_y_tr_fnn == 1).sum().item()
_n_nonfraud_fnn = (_y_tr_fnn == 0).sum().item()
_pos_weight_fnn = torch.tensor([_n_nonfraud_fnn / _n_fraud_fnn], device=_dev_fnn)
print(f"\nClass imbalance handling:")
print(f"  Fraud: {_n_fraud_fnn:,}, Non-fraud: {_n_nonfraud_fnn:,}")
print(f"  Positive class weight: {_pos_weight_fnn.item():.2f}")


class _FNN(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 1)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        x = F.relu(self.fc1(x)); x = self.dropout(x)
        x = F.relu(self.fc2(x)); x = self.dropout(x)
        return self.fc3(x).squeeze(-1)

_model_fnn = _FNN(input_dim=_d_fnn, hidden1=128, hidden2=64).to(_dev_fnn)
_opt_fnn   = _Adam(_model_fnn.parameters(), lr=1e-3, weight_decay=1e-5)

print(f"\nModel : FNN(in={_d_fnn}, hidden=[128, 64], out=1)")
print(f"  Total parameters : {sum(p.numel() for p in _model_fnn.parameters()):,}")
_EPOCHS_FNN   = 50
_BATCH_SIZE_F = 512
_model_fnn.train()
print(f"\nTraining FNN for {_EPOCHS_FNN} epochs on {_N_tr_fnn:,} rows (batch={_BATCH_SIZE_F})...")

for ep in range(_EPOCHS_FNN):
    _perm       = torch.randperm(_N_tr_fnn)
    _epoch_loss = 0.0
    _n_batches  = 0
    for i in range(0, _N_tr_fnn, _BATCH_SIZE_F):
        _bi = _perm[i:i + _BATCH_SIZE_F]
        _opt_fnn.zero_grad()
        _logits = _model_fnn(_X_tr_fnn[_bi])
        _loss   = F.binary_cross_entropy_with_logits(
            _logits, _y_tr_fnn[_bi].float(), pos_weight=_pos_weight_fnn)
        _loss.backward()
        _opt_fnn.step()
        _epoch_loss += _loss.item(); _n_batches += 1
    if (ep + 1) % 10 == 0 or ep == 0:
        _model_fnn.eval()
        with torch.no_grad():
            _p_tr  = torch.sigmoid(_model_fnn(_X_tr_fnn)).cpu().numpy()
            _pr_tr = (_p_tr >= 0.5).astype(int)
            _yt_np = _y_tr_fnn.cpu().numpy()
            _acc   = (_pr_tr == _yt_np).mean()
            _rec   = ((_pr_tr == 1) & (_yt_np == 1)).sum() / max((_yt_np == 1).sum(), 1)
        _model_fnn.train()
        print(f"  Epoch {ep+1:3d} | Loss={_epoch_loss/_n_batches:.4f} | Acc={_acc:.4f} | Recall={_rec:.4f}")

print("\nTraining complete.")

_model_fnn.eval()
with torch.no_grad():
    _probs_te_fnn = torch.sigmoid(_model_fnn(_X_te_fnn)).cpu().numpy()
    _preds_te_fnn = (_probs_te_fnn >= 0.5).astype(int)

_y_te_np      = _y_te_fnn.cpu().numpy()

print("FNN Test Results")
print(f"  Accuracy  : {accuracy_score(_y_te_np, _preds_te_fnn):.4f}")
print(f"  Precision : {precision_score(_y_te_np, _preds_te_fnn, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(_y_te_np, _preds_te_fnn, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(_y_te_np, _preds_te_fnn, zero_division=0):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(_y_te_np, _probs_te_fnn) if len(np.unique(_y_te_np)) > 1 else 0.0:.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(_y_te_np, _preds_te_fnn)}")


print("Calibrating on original (unbalanced) training data...")
with torch.no_grad():
    _raw_scores_o_fnn = torch.sigmoid(_model_fnn(_Xo_fnn)).cpu().numpy().ravel()

_cal_lr_fnn = _CalLR(max_iter=500, random_state=RANDOM_STATE)
_cal_lr_fnn.fit(_raw_scores_o_fnn.reshape(-1, 1), _yo_fnn)
_calibrated_probs_te_fnn = _cal_lr_fnn.predict_proba(_probs_te_fnn.reshape(-1, 1))[:, 1]

_fpr_c, _tpr_c, _thr_c = roc_curve(_y_te_np, _calibrated_probs_te_fnn)
_best_thr_fnn  = float(_thr_c[np.argmax(_tpr_c - _fpr_c)])
_preds_cal_fnn = (_calibrated_probs_te_fnn >= _best_thr_fnn).astype(int)

print(f"\nCalibrated Results (threshold = {_best_thr_fnn:.4f}):")
print(f"  Accuracy  : {accuracy_score(_y_te_np, _preds_cal_fnn):.4f}")
print(f"  Precision : {precision_score(_y_te_np, _preds_cal_fnn, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(_y_te_np, _preds_cal_fnn, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(_y_te_np, _preds_cal_fnn, zero_division=0):.4f}")
print(f"  AUC-ROC   : {sk_auc(_fpr_c, _tpr_c):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(_y_te_np, _preds_cal_fnn)}")

print("\n✓ FNN training and evaluation complete.")

In [ ]:
# DNN
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression as _CalLR
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.metrics import auc as sk_auc

def _np32(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('float32')
def _np64(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('int64')

class _Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        self.params = [p for p in params if p.requires_grad]
        self.lr = lr; self.b1, self.b2 = betas
        self.eps = eps; self.wd = weight_decay; self.t = 0
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_(); p.grad.zero_()
    def step(self):
        self.t += 1
        bc1, bc2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t
        for i, p in enumerate(self.params):
            if p.grad is None: continue
            g = p.grad.data
            if self.wd: g = g.add(p.data, alpha=self.wd)
            self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
            self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            p.data.addcdiv_(self.m[i] / bc1, (self.v[i] / bc2).sqrt().add_(self.eps), value=-self.lr)

RANDOM_STATE = 42
if 'X_train_orig_scaled' not in dir() or X_train_orig_scaled is None:
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )

print("Section 14.1i  Deep Neural Network (DNN)")

print(f"  PyTorch {torch.__version__}")

_dev_dnn = (torch.device("mps")  if torch.backends.mps.is_available() else
            torch.device("cuda") if torch.cuda.is_available() else
            torch.device("cpu"))
print(f"  Device : {_dev_dnn}")

_DNN_CAP = 30_000
_X_tr_full_dnn = _np32(X_train_scaled)
_y_tr_full_dnn = _np64(y_train_balanced)

if len(_y_tr_full_dnn) > _DNN_CAP:
    _rng_dnn  = np.random.RandomState(RANDOM_STATE + 6)
    _fr_idx_d = np.where(_y_tr_full_dnn == 1)[0]
    _nf_idx_d = np.where(_y_tr_full_dnn == 0)[0]
    _n_fr_d   = min(len(_fr_idx_d), _DNN_CAP // 2)
    _fr_kp_d  = (_rng_dnn.choice(_fr_idx_d, _n_fr_d, replace=False)
                 if len(_fr_idx_d) > _n_fr_d else _fr_idx_d)
    _nf_kp_d  = _rng_dnn.choice(_nf_idx_d, _DNN_CAP - len(_fr_kp_d), replace=False)
    _keep_d   = np.sort(np.concatenate([_fr_kp_d, _nf_kp_d]))
    _X_tr_full_dnn = _X_tr_full_dnn[_keep_d]
    _y_tr_full_dnn = _y_tr_full_dnn[_keep_d]

_X_tr_dnn = torch.tensor(_X_tr_full_dnn,              dtype=torch.float32, device=_dev_dnn)
_y_tr_dnn = torch.tensor(_y_tr_full_dnn,              dtype=torch.long,    device=_dev_dnn)
_X_te_dnn = torch.tensor(_np32(X_test_scaled),        dtype=torch.float32, device=_dev_dnn)
_y_te_dnn = torch.tensor(_np64(y_test),               dtype=torch.long,    device=_dev_dnn)
_Xo_dnn   = torch.tensor(_np32(X_train_orig_scaled),  dtype=torch.float32, device=_dev_dnn)
_yo_dnn   = _np64(y_train)

_N_tr_dnn, _d_dnn = _X_tr_dnn.shape
_N_te_dnn = _X_te_dnn.shape[0]

print(f"\nTraining set (capped) : {_N_tr_dnn:,}  "
      f"(fraud: {(_y_tr_dnn==1).sum().item():,} | non-fraud: {(_y_tr_dnn==0).sum().item():,})")
print(f"Test set              : {_N_te_dnn:,}  (fraud: {(_y_te_dnn==1).sum().item():,})")

_n_fraud_dnn    = (_y_tr_dnn == 1).sum().item()
_n_nonfraud_dnn = (_y_tr_dnn == 0).sum().item()
_pos_weight_dnn = torch.tensor([_n_nonfraud_dnn / _n_fraud_dnn], device=_dev_dnn)
print(f"\nClass imbalance handling:")
print(f"  Fraud: {_n_fraud_dnn:,}, Non-fraud: {_n_nonfraud_dnn:,}")
print(f"  Positive class weight: {_pos_weight_dnn.item():.2f}")

class _DNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128, 64, 32]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev_dim, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.3)]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(-1)

_model_dnn = _DNN(input_dim=_d_dnn, hidden_dims=[256, 128, 64, 32]).to(_dev_dnn)
_opt_dnn   = _Adam(_model_dnn.parameters(), lr=1e-3, weight_decay=1e-5)

print(f"\nModel : DNN(in={_d_dnn}, hidden=[256, 128, 64, 32], out=1)")
print(f"  Total parameters : {sum(p.numel() for p in _model_dnn.parameters()):,}")

_EPOCHS_DNN     = 50
_BATCH_SIZE_DNN = 512
_model_dnn.train()
print(f"\nTraining DNN for {_EPOCHS_DNN} epochs on {_N_tr_dnn:,} rows (batch={_BATCH_SIZE_DNN})...")

for ep in range(_EPOCHS_DNN):
    _perm       = torch.randperm(_N_tr_dnn)
    _epoch_loss = 0.0
    _n_batches  = 0
    for i in range(0, _N_tr_dnn, _BATCH_SIZE_DNN):
        _bi = _perm[i:i + _BATCH_SIZE_DNN]
        _opt_dnn.zero_grad()
        _logits = _model_dnn(_X_tr_dnn[_bi])
        _loss   = F.binary_cross_entropy_with_logits(
            _logits, _y_tr_dnn[_bi].float(), pos_weight=_pos_weight_dnn)
        _loss.backward()
        _opt_dnn.step()
        _epoch_loss += _loss.item(); _n_batches += 1
    if (ep + 1) % 10 == 0 or ep == 0:
        _model_dnn.eval()
        with torch.no_grad():
            _p_tr  = torch.sigmoid(_model_dnn(_X_tr_dnn)).cpu().numpy()
            _pr_tr = (_p_tr >= 0.5).astype(int)
            _yt_np = _y_tr_dnn.cpu().numpy()
            _acc   = (_pr_tr == _yt_np).mean()
            _rec   = ((_pr_tr == 1) & (_yt_np == 1)).sum() / max((_yt_np == 1).sum(), 1)
        _model_dnn.train()
        print(f"  Epoch {ep+1:3d} | Loss={_epoch_loss/_n_batches:.4f} | Acc={_acc:.4f} | Recall={_rec:.4f}")

print("\nTraining complete.")

_model_dnn.eval()
with torch.no_grad():
    _probs_te_dnn = torch.sigmoid(_model_dnn(_X_te_dnn)).cpu().numpy()
    _preds_te_dnn = (_probs_te_dnn >= 0.5).astype(int)

_y_te_np_dnn = _y_te_dnn.cpu().numpy()

print("DNN Test Results")
print(f"  Accuracy  : {accuracy_score(_y_te_np_dnn, _preds_te_dnn):.4f}")
print(f"  Precision : {precision_score(_y_te_np_dnn, _preds_te_dnn, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(_y_te_np_dnn, _preds_te_dnn, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(_y_te_np_dnn, _preds_te_dnn, zero_division=0):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(_y_te_np_dnn, _probs_te_dnn) if len(np.unique(_y_te_np_dnn)) > 1 else 0.0:.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(_y_te_np_dnn, _preds_te_dnn)}")
print("Calibrating on original (unbalanced) training data...")
with torch.no_grad():
    _raw_scores_o_dnn = torch.sigmoid(_model_dnn(_Xo_dnn)).cpu().numpy().ravel()

_cal_lr_dnn = _CalLR(max_iter=500, random_state=RANDOM_STATE)
_cal_lr_dnn.fit(_raw_scores_o_dnn.reshape(-1, 1), _yo_dnn)
_calibrated_probs_te_dnn = _cal_lr_dnn.predict_proba(_probs_te_dnn.reshape(-1, 1))[:, 1]

_fpr_cd, _tpr_cd, _thr_cd = roc_curve(_y_te_np_dnn, _calibrated_probs_te_dnn)
_best_thr_dnn  = float(_thr_cd[np.argmax(_tpr_cd - _fpr_cd)])
_preds_cal_dnn = (_calibrated_probs_te_dnn >= _best_thr_dnn).astype(int)

print(f"\nCalibrated Results (threshold = {_best_thr_dnn:.4f}):")
print(f"  Accuracy  : {accuracy_score(_y_te_np_dnn, _preds_cal_dnn):.4f}")
print(f"  Precision : {precision_score(_y_te_np_dnn, _preds_cal_dnn, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(_y_te_np_dnn, _preds_cal_dnn, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(_y_te_np_dnn, _preds_cal_dnn, zero_division=0):.4f}")
print(f"  AUC-ROC   : {sk_auc(_fpr_cd, _tpr_cd):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(_y_te_np_dnn, _preds_cal_dnn)}")

print("\n✓ DNN training and evaluation complete.")

### 17.3 Hyperparameter Search Results

#### Final Model Results (Run 2 — Refined Search Spaces)

| Model | Section | Imbalance Strategy | Search Method | Max Evals | CV Folds | CV ROC-AUC | Test ROC-AUC | Test Recall | Fraud Detected (54 total) | Youden Threshold |
|-------|---------|-------------------|--------------|-----------|----------|------------|-------------|-------------|--------------------------|-----------------|
| **Logistic Regression** | 10 | L2 regularization (`C=1.0`) | No search (fixed) | — | — | — | 0.7147 | **66.7%** | **36** | 0.50 (default) |
| **Random Forest** | 11.1 | `class_weight='balanced_subsample'` | Hyperopt TPE | 60 | 3-fold | 0.7509 | 0.7435 | 42.6% | 23 | 0.000315 |
| **XGBoost** | 11.2 | `scale_pos_weight = 5822.75` | Hyperopt TPE | 60 | 3-fold | 0.7524 | **0.7481** | 35.2% | 19 | 0.000335 |
| **LightGBM** | 11.3 | `class_weight='balanced'` | Hyperopt TPE | 60 | 3-fold | 0.7384 | 0.6017 | 27.8% | 15 | 0.000271 |
| **HistGradBoost** | 11.4 | `class_weight='balanced'` | Hyperopt TPE | 60 | 3-fold | 0.7376 | 0.6965 | 40.7% | 22 | 0.000309 |
| **SVM (LinearSVC)** | 11.5 | `class_weight='balanced'` | Hyperopt TPE | 60 | 3-fold | 0.7356 | 0.7288 | 53.7% | 29 | 0.000206 |
| **MLP (Neural Net)** | 11.6 | `compute_sample_weight('balanced')` | Hyperopt TPE | 40 | 3-fold | 0.5519 | 0.6524 | 48.1% | 26 | 0.000177 |

---

#### Best Hyperparameters (Run 2)

| Model | Best Hyperparameters |
|-------|---------------------|
| **Logistic Regression** | `max_iter=1000`, `C=1.0`, `solver='saga'`, `class_weight='balanced'` |
| **Random Forest** | `n_estimators=300`, `max_depth=3`, `max_features=0.6`, `max_samples=0.7`, `min_samples_leaf=4`, `min_samples_split=14` |
| **XGBoost** | `n_estimators=550`, `max_depth=2`, `learning_rate=0.011`, `subsample=0.583`, `colsample_bytree=0.891`, `min_child_weight=17`, `gamma=0.118`, `reg_alpha=1.943`, `reg_lambda=1.066` |
| **LightGBM** | `n_estimators=200`, `max_depth=4`, `num_leaves=28`, `learning_rate=0.038`, `subsample=0.860`, `colsample_bytree=0.963`, `min_child_samples=180`, `lambda_l1=0.593`, `lambda_l2=49.1`, `min_gain_to_split=0.0042` |
| **HistGradBoost** | `max_iter=350`, `max_depth=10`, `learning_rate=0.0073`, `max_leaf_nodes=15`, `min_samples_leaf=90`, `l2_regularization=1.70`, `max_bins=128` |
| **SVM** | `C=0.00632` |
| **MLP** | `hidden_layer_sizes=(64, 64, 32)`, `alpha=0.000167`, `learning_rate_init=0.000326`, `batch_size=256` |

---

#### Run 1 vs Run 2 — Performance Comparison


| Model | Run 1 Test AUC | Run 1 Recall | Run 1 Fraud Det. | Run 2 Test AUC | Run 2 Recall | Run 2 Fraud Det. | Δ AUC | Δ Recall |
|-------|---------------|-------------|-----------------|---------------|-------------|-----------------|-------|---------|
| **Logistic Regression** | 0.7147 | 66.7% | 36/54 | 0.7147 | 66.7% | 36/54 | 0.000 | 0.0 pp |
| **Random Forest** | 0.7245 | 38.9% | 21/54 | **0.7435** | **42.6%** | **23/54** | 🟢 +0.019 | 🟢 +3.7 pp |
| **XGBoost** | 0.6404 | 25.9% | 14/54 | **0.7481** | 35.2% | 19/54 | 🟢 +0.108 | 🟢 +9.3 pp |
| **LightGBM** | 0.5958 | 24.1% | 13/54 | 0.6017 | 27.8% | 15/54 | 🟢 +0.006 | 🟢 +3.7 pp |
| **HistGradBoost** | 0.6718 | 24.1% | 13/54 | **0.6965** | **40.7%** | **22/54** | 🟢 +0.025 | 🟢 +16.7 pp |
| **SVM (LinearSVC)** | **0.7288** | **53.7%** | **29/54** | 0.7288 | 53.7% | 29/54 | 0.000 | 0.0 pp |
| **MLP (Neural Net)** | 0.5632 | 40.7% | 22/54 | 0.6524 | 48.1% | 26/54 | 🟢 +0.089 | 🟢 +7.4 pp |
